# NB04 V3_1 — Full satellite transfer experiment

Uses the **frozen M2 warm-start full-mosaic maps** and **previously downloaded V3 satellite features**. No Earth Engine calls or new downloads. Native UAV probabilities are copied to the Colab SSD; all aggregations and feature alignment occur locally. Per-site outputs, samples and held-out predictions are cached on Drive.

**Targets**: `mean_probability` = average of valid UAV pixel probabilities (no classification threshold); `cover_fraction_p05` = share of valid UAV pixels with p≥0.5. These are distinct targets. We report both. **Primary validation**: leave-one-site-out (LOSO), 4 forests. Scores quantify transfer to UAV-derived labels, not ground-truth validation of satellite maps.


## 00 — USER SETTINGS

In [ ]:
# ==================== USER SETTINGS ====================
SCALES_M = [10,30,50,75,100,125]
FEATURE_SETS = ['ANNUAL_S2','EMBEDDING','S2_PLUS_EMBEDDING']
MODELS = ['RIDGE','RF','EXTRA_TREES','HIST_GB']
TARGETS = ['mean_probability','cover_fraction_p05']
MAX_SAMPLES_PER_MOSAIC = 8000
MIN_UAV_COVERAGE = .90
SEED = 20261022
COPY_UAV_TO_SSD = True
COPY_FEATURES_TO_SSD = True
FORCE_REBUILD = False  # keep False: resumes cached aggregate/sample/fold files
N_JOBS = 2
SAVE_FIGURES = True
# ========================================================

# BLOCK 06 — targeted high-cover recovery (independent of full Block 04 rerun)
B06_RUN = True
B06_SCALES_M = [75,100,125]
B06_FEATURE_SETS = ['EMBEDDING','S2_PLUS_EMBEDDING']
B06_TARGETS = ['mean_probability','cover_fraction_p05']
B06_WEIGHT_STRENGTHS = [0,2,5]  # 0=standard boosting; >0 emphasizes cover above 20%
B06_HIGH_THRESHOLD = .40
B06_REBUILD = False


# ========== BLOCKS 07–08: ROI seasonal LOSO settings ==========
from pathlib import Path
ROOT=Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
FINAL=ROOT/"Final"
ROI_PATH=ROOT/"data/Orthoo_RGB/ROI.shp"
SCALES=[75,100,125]
TARGETS=["mean_probability"]  # optional: add "cover_fraction_p05"
FEATURE_SETS={
 "EMB":["EMB_ANNUAL"],
 "QUARTERLY":["S2_QUARTERLY_3YR"],
 "EMB_QUARTERLY":["EMB_ANNUAL","S2_QUARTERLY_3YR"],
 "EMB_ANNUAL_QUARTERLY":["EMB_ANNUAL","RAW_ANNUAL","S2_QUARTERLY_3YR"]
}
CV_SCHEMES=["FIVE_MOSAICS","FOUR_FORESTS"]
MAX_CELLS_PER_MOSAIC=8000
MIN_ROI_COVERAGE=0.90    # fractional pixel coverage inside ROI
ROI_SUBPIXELS=4          # 4x4 supersampling
MIN_UAV_COVERAGE=0.90
MIN_QUARTER_CLEAR=1
SEED=20261022
WEIGHT_STRENGTHS=[0,2]
HIGH_COVER_THRESHOLD=0.40
FORCE_SAMPLES=False
FORCE_FOLDS=False
SAVE_PLOTS=True


## 01 — Mount Drive, audit cached inputs and stage them on local SSD

In [ ]:
%pip -q install rasterio geopandas pandas pyarrow scikit-learn matplotlib scipy
import os, json, math, time, hashlib, shutil, gc, warnings
from pathlib import Path
import numpy as np, pandas as pd, rasterio
from affine import Affine
from rasterio.windows import Window
from rasterio.warp import reproject, Resampling
from sklearn.ensemble import RandomForestRegressor,ExtraTreesRegressor,HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert ROOT is not None
OLD=ROOT/'Outputs/V3/04_satellite_Musanga/features'
MAPS=ROOT/'Outputs/V3_1/03_M2_warmstart_c6a33504c37efb2ebb84_full_mosaic/probability_maps'
OUT=ROOT/'Outputs/V3_1/04_satellite_full_warmstart'
CACHE=OUT/'cache';AGG=CACHE/'aggregates';SAMPLE=CACHE/'samples';FOLDS=OUT/'folds';FIG=OUT/'figures'
for p in (CACHE,AGG,SAMPLE,FOLDS,FIG):p.mkdir(parents=True,exist_ok=True)
SSD=Path('/content/musanga_nb04_ssd');(SSD/'uav').mkdir(parents=True,exist_ok=True);(SSD/'features').mkdir(exist_ok=True)
YEARS={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2020}
SITE_MOSAICS={'Baego':['Baego.tif'],
 'MOS':['Mos_1_2_transparent_mosaic_group1_georef.tif','Mos_5_6_transparent_mosaic_group1_georef.tif'],
 'Yangambi':['Yangambi_georef.tif'],'Yoko':['Yoko_kvo_georef.tif']}
FAMILY_NAMES={'ANNUAL_S2':['RAW_ANNUAL'],'EMBEDDING':['EMB_ANNUAL'],
              'S2_PLUS_EMBEDDING':['RAW_ANNUAL','EMB_ANNUAL']}
def uavfile(name):return MAPS/f'{Path(name).stem}_M2_warmstart_epoch02_probability_u16_FULL.tif'
def featfile(site,fam):return OLD/fam/f'{site}_{fam}_{YEARS[site]}_10m.tif'
def staged(path,folder,copy):
    assert path.is_file(),('Missing required cached file',path)
    target=folder/path.name
    if not copy:return path
    srcstat=path.stat()
    if not target.exists() or target.stat().st_size!=srcstat.st_size:
        print('COPY to SSD:',path.name,round(srcstat.st_size/1e9,2),'GB',flush=True)
        tmp=target.with_name(target.name+'.partial')
        shutil.copyfile(path,tmp);os.replace(tmp,target)
    return target
UAV={name:staged(uavfile(name),SSD/'uav',COPY_UAV_TO_SSD) for ns in SITE_MOSAICS.values() for name in ns}
FAMILY={'RAW_ANNUAL','EMB_ANNUAL'}
assert all(featfile(site,fam).is_file() for site in SITE_MOSAICS for fam in FAMILY)
FEATURES={(site,fam):staged(featfile(site,fam),SSD/'features',COPY_FEATURES_TO_SSD)
          for site in SITE_MOSAICS for fam in sorted(FAMILY)}
print('Input audit: ',len(UAV),'UAV maps,',len(FEATURES),'cached predictor stacks')
print('SSD free GB:',round(shutil.disk_usage(SSD).free/1e9,2),flush=True)


## 02 — Aggregate *entire* native probability mosaics to 10–125 m

Area-weighted aggregation onto the **same year-matched Sentinel-2 10 m grid origin**, scaled to 10, 30, 50, 75, 100, 125 m. Intermediate rasters are float32 `[mean_probability, cover_fraction_p05, valid_UAV_coverage]`. No overlapping-mosaic double counting: mosaic cells remain separately indexed. Computing at 75/125 m uses a valid satellite-grid transform and area resampling, not naive array reshaping. Each mosaic/scale is cached on Drive.


In [ ]:
def grid_for(site,scale):
    with rasterio.open(FEATURES[(site,'RAW_ANNUAL')]) as ds:
        assert abs(ds.transform.a-10)<1e-5 and abs(ds.transform.e+10)<1e-5
        f=scale/10.
        return dict(height=math.ceil(ds.height/f),width=math.ceil(ds.width/f),
                    transform=ds.transform*Affine.scale(f,f),crs=ds.crs)
def profile_for(g,count):
    return dict(driver='GTiff',height=g['height'],width=g['width'],count=count,
                crs=g['crs'],transform=g['transform'],dtype='float32',nodata=-9999.,
                tiled=True,blockxsize=256,blockysize=256,compress='deflate',predictor=3,bigtiff='YES')
def warp(src,g,rs=Resampling.average,srcnodata=None):
    result=np.full((g['height'],g['width']),-9999.,np.float32)
    reproject(source=src,destination=result,src_nodata=srcnodata,
              dst_transform=g['transform'],dst_crs=g['crs'],dst_nodata=-9999.,
              resampling=rs,num_threads=2)
    return result
def aggregate_one(site,name,scale):
    dst=AGG/f'{Path(name).stem}_{scale}m.tif'
    if dst.exists() and not FORCE_REBUILD:return dst
    g=grid_for(site,scale)
    with rasterio.open(UAV[name]) as src:
        assert src.count==1 and src.nodata==65535
        # Reproject native probabilities / native hard classes separately; construct
        # binary and valid auxiliary rasters in SSD with block streaming.
        scratch=SSD/f'{Path(name).stem}_native_derived.tif'
        pr=src.profile.copy()
        for key in ('photometric','alpha','extrasamples','NBITS','colorinterp'):pr.pop(key,None)
        pr.update(driver='GTiff',count=3,dtype='float32',nodata=-9999.,
                  tiled=True,blockxsize=256,blockysize=256,compress='deflate',predictor=3)
        with rasterio.open(scratch,'w',**pr) as aux:
            for _,win in src.block_windows(1):
                a=src.read(1,window=win);ok=a!=65535
                prob=np.where(ok,a.astype('float32')/10000.,-9999.).astype('float32')
                hard=np.where(ok,(a>=5000).astype('float32'),-9999.).astype('float32')
                valid=ok.astype('float32')
                aux.write(prob,1,window=win);aux.write(hard,2,window=win);aux.write(valid,3,window=win)
        with rasterio.open(scratch) as aux:
            vals=[warp(rasterio.band(aux,i),g,srcnodata=(-9999. if i<3 else None)) for i in (1,2,3)]
        scratch.unlink(missing_ok=True)
    tmp=dst.with_suffix('.partial.tif')
    with rasterio.open(tmp,'w',**profile_for(g,3)) as ds:
        for i,a in enumerate(vals,1):ds.write(a,i)
        ds.set_band_description(1,'mean_probability')
        ds.set_band_description(2,'cover_fraction_p05')
        ds.set_band_description(3,'uav_valid_coverage')
        ds.update_tags(PRODUCTION_MODEL='M2_warmstart_epoch02',AGGREGATION_M=str(scale),
                       PROB_THRESHOLD='0.5',SOURCE_MOSAIC=name,COMPLETE='YES')
    os.replace(tmp,dst)
    return dst
AGG_LOOKUP={}
for scale in SCALES_M:
    t=time.perf_counter()
    for site,names in SITE_MOSAICS.items():
        for name in names:
            AGG_LOOKUP[(name,scale)]=aggregate_one(site,name,scale)
            print('AGG READY',scale,'m',site,Path(name).stem,'elapsed',round(time.perf_counter()-t,1),'s',flush=True)
print('All requested scale rasters cached.')


## 03 — Build consistent satellite/UAV samples, cache per resolution

Use all available cached S2 annual bands and all 64 embedding dimensions. Sample up to the configured maximum cells **per mosaic**, with at least 90% valid native UAV coverage. Preserve target and satellite-band missingness explicitly; model-specific feature filtering happens at fit time. Samples are reproducibly seeded. Full mosaics may overlap; sampled mosaic membership is retained.


In [ ]:
def features_on_grid(site,scale):
    g=grid_for(site,scale); cubes=[]; cols=[]
    for family in ('RAW_ANNUAL','EMB_ANNUAL'):
        with rasterio.open(FEATURES[(site,family)]) as ds:
            assert ds.count>0 and ds.crs==g['crs']
            for bi in range(1,ds.count+1):
                a=warp(rasterio.band(ds,bi),g,srcnodata=ds.nodata)
                a[a==-9999]=np.nan
                cubes.append(a);cols.append(f'{family}__{bi:02d}')
    return np.stack(cubes),cols
ALL_SAMPLES={}
for scale in SCALES_M:
    cache=SAMPLE/f'cells_{scale}m.parquet'
    if cache.is_file() and not FORCE_REBUILD:
        frame=pd.read_parquet(cache);ALL_SAMPLES[scale]=frame
        print('SAMPLE CACHE HIT',scale,'m',len(frame),'rows',flush=True);continue
    rows=[]
    for site,names in SITE_MOSAICS.items():
        g=grid_for(site,scale);cube,cols=features_on_grid(site,scale)
        for name in names:
            with rasterio.open(AGG_LOOKUP[(name,scale)]) as ds:
                y1,y2,valid=ds.read()
                assert ds.crs==g['crs'] and ds.transform.almost_equals(g['transform'])
            ok=(valid>=MIN_UAV_COVERAGE)&np.isfinite(y1)&np.isfinite(y2)&(y1>=0)&(y1<=1)&(y2>=0)&(y2<=1)
            # Drop missing feature cells only for the common complete-case benchmark
            ok &= np.all(np.isfinite(cube),axis=0)
            rr,cc=np.where(ok)
            rng=np.random.default_rng(SEED+sum(map(ord,name))+scale)
            take=rng.choice(len(rr),size=min(len(rr),MAX_SAMPLES_PER_MOSAIC),replace=False)
            rr=rr[take];cc=cc[take]
            frame=pd.DataFrame(cube[:,rr,cc].T,columns=cols)
            frame.insert(0,'mosaic',name);frame.insert(0,'site',site)
            frame.insert(2,'row',rr);frame.insert(3,'col',cc)
            frame['mean_probability']=y1[rr,cc];frame['cover_fraction_p05']=y2[rr,cc]
            rows.append(frame)
            print('SAMPLED',scale,'m',site,len(frame),'cells',flush=True)
        del cube;gc.collect()
    frame=pd.concat(rows,ignore_index=True)
    assert all((frame.site==s).sum()>0 for s in SITE_MOSAICS)
    frame.to_parquet(cache,index=False);ALL_SAMPLES[scale]=frame
    print('SAVED',cache,len(frame),'rows',flush=True)


## 04 — Multi-model leave-one-site-out benchmark (resumable, live progress)

Four regressors: regularized linear **Ridge**, **Random Forest**, **Extra Trees**, and **Histogram Gradient Boosting**. Fit each without using the held-out forest. Compare three predictor sets and both continuous targets. A **training-mean baseline** is included. Save *each completed fold* as a separate CSV so a Colab interruption does not discard earlier work. Fits run on CPU; GPU is not needed.


In [ ]:
def estimator(name):
    if name=='RIDGE':return make_pipeline(StandardScaler(),Ridge(alpha=20.))
    if name=='RF':return RandomForestRegressor(n_estimators=150,max_features=.65,
        min_samples_leaf=5,n_jobs=N_JOBS,random_state=SEED)
    if name=='EXTRA_TREES':return ExtraTreesRegressor(n_estimators=150,max_features=.65,
        min_samples_leaf=5,n_jobs=N_JOBS,random_state=SEED)
    if name=='HIST_GB':return HistGradientBoostingRegressor(learning_rate=.06,max_iter=200,
        max_leaf_nodes=20,min_samples_leaf=40,l2_regularization=2,
        early_stopping=False,random_state=SEED)
    raise ValueError(name)
def metrics(y,p):
    return dict(R2=r2_score(y,p),MAE_pp=100*mean_absolute_error(y,p),
                RMSE_pp=100*np.sqrt(mean_squared_error(y,p)),bias_pp=100*np.mean(p-y))
results=[];commenced=time.perf_counter()
for scale in SCALES_M:
    frame=ALL_SAMPLES[scale]
    for feature_set in FEATURE_SETS:
        fams=FAMILY_NAMES[feature_set]
        cols=[c for c in frame if any(c.startswith(f+'__') for f in fams)]
        for target in TARGETS:
            for heldout in SITE_MOSAICS:
                train=frame[frame.site!=heldout];test=frame[frame.site==heldout]
                assert len(train)>100 and len(test)>10
                Xtr=train[cols].to_numpy('float32');Xte=test[cols].to_numpy('float32')
                ytr=train[target].to_numpy('float32');yte=test[target].to_numpy('float32')
                for model_name in MODELS+['TRAIN_MEAN']:
                    key=f'{scale}m_{feature_set}_{target}_{heldout}_{model_name}'
                    dest=FOLDS/(key+'.csv')
                    if dest.is_file() and not FORCE_REBUILD:
                        fold=pd.read_csv(dest)
                        print('CACHE HIT',key,flush=True)
                    else:
                        t=time.perf_counter()
                        print('START',key,'train',len(ytr),'test',len(yte),flush=True)
                        if model_name=='TRAIN_MEAN':p=np.repeat(ytr.mean(),len(yte))
                        else:
                            m=estimator(model_name);m.fit(Xtr,ytr)
                            p=np.clip(m.predict(Xte),0,1)
                        fold=test[['site','mosaic','row','col']].copy()
                        fold['scale_m']=scale;fold['feature_set']=feature_set
                        fold['target']=target;fold['model']=model_name
                        fold['observed']=yte;fold['predicted']=p
                        fold.to_csv(dest,index=False)
                        met=metrics(yte,p)
                        print('DONE',key,'MAE',round(met['MAE_pp'],2),'pp',
                              'R2',round(met['R2'],3),'seconds',round(time.perf_counter()-t,1),flush=True)
                    assert len(fold)==len(yte)
                    scores=metrics(fold.observed.to_numpy(),fold.predicted.to_numpy())
                    results.append(dict(scale_m=scale,feature_set=feature_set,target=target,
                        heldout_site=heldout,model=model_name,n_test=len(fold),**scores))
    summary=pd.DataFrame(results)
    summary.to_csv(OUT/'04_LOSO_metrics_progress.csv',index=False)
    print('FINISHED scale',scale,'m; elapsed',round((time.perf_counter()-commenced)/60,1),'minutes',flush=True)
summary=pd.DataFrame(results)
summary.to_csv(OUT/'04_LOSO_metrics_full.csv',index=False)
print('=== LOSO METRICS BY SITE ===')
display(summary.round(3))
print('=== MEAN ACROSS HELD-OUT SITES (EQUAL SITE WEIGHTS) ===')
ranked=summary.groupby(['target','scale_m','feature_set','model'],as_index=False).agg(
    MAE_pp=('MAE_pp','mean'),RMSE_pp=('RMSE_pp','mean'),R2=('R2','mean'),
    bias_pp=('bias_pp','mean'))
display(ranked.sort_values(['target','MAE_pp']).round(3))
ranked.to_csv(OUT/'04_LOSO_ranked_equal_site.csv',index=False)


## 05 — Figures, diagnostic scatterplots and best-model recommendations

Show mean LOSO MAE versus scale and predictor set, and four held-out-site scatterplots for the lowest-MAE configuration separately for each UAV target. Save figures and per-site predictions; report whether satellite models beat the training-mean baseline. **Do not interpret in-sample circle scores as validation of satellite generalization.**


In [ ]:
from IPython.display import display
assert (OUT/'04_LOSO_metrics_full.csv').exists()
summary=pd.read_csv(OUT/'04_LOSO_metrics_full.csv')
ranked=pd.read_csv(OUT/'04_LOSO_ranked_equal_site.csv')
for target in TARGETS:
    entries=ranked[(ranked.target==target)&(ranked.model!='TRAIN_MEAN')].sort_values('MAE_pp')
    winner=entries.iloc[0]
    print('\nBEST',target,'scale',winner.scale_m,'m','features',winner.feature_set,
          'model',winner.model,'mean heldout-site MAE',round(winner.MAE_pp,3),'pp',flush=True)
    baseline=ranked[(ranked.target==target)&(ranked.model=='TRAIN_MEAN')&(ranked.scale_m==winner.scale_m)]
    if len(baseline):print('Training-mean baseline MAE',round(float(baseline.MAE_pp.iloc[0]),3),'pp')
    fig,axes=plt.subplots(2,2,figsize=(11,9),constrained_layout=True)
    for ax,site in zip(axes.flat,SITE_MOSAICS):
        path=FOLDS/f'{int(winner.scale_m)}m_{winner.feature_set}_{target}_{site}_{winner.model}.csv'
        df=pd.read_csv(path)
        ax.scatter(100*df.observed,100*df.predicted,s=4,alpha=.15,rasterized=True)
        ax.plot([0,100],[0,100],'--',color='grey',linewidth=1)
        metric=metrics(df.observed.to_numpy(),df.predicted.to_numpy())
        ax.set(xlim=(0,100),ylim=(0,100),title=f'{site} | MAE {metric["MAE_pp"]:.2f} pp | R² {metric["R2"]:.2f}',
               xlabel='UAV observed target (%)',ylabel='Satellite predicted target (%)')
    fig.suptitle(f'LOSO satellite: {target} | {winner.feature_set}, {int(winner.scale_m)} m, {winner.model}')
    if SAVE_FIGURES:fig.savefig(FIG/f'best_LOSO_scatter_{target}.png',dpi=175)
    plt.show();plt.close(fig)
    subset=ranked[(ranked.target==target)&(ranked.model!='TRAIN_MEAN')]
    fig,ax=plt.subplots(figsize=(10,5))
    for (features,model),g in subset.groupby(['feature_set','model']):
        ax.plot(g.scale_m,g.MAE_pp,'-o',markersize=3,label=f'{features} / {model}')
    ax.set(xlabel='Satellite grid resolution (m)',ylabel='LOSO mean MAE (percentage points)',
           title=f'{target}: out-of-site performance by resolution and model')
    ax.grid(alpha=.2);ax.legend(fontsize=7,ncol=3)
    if SAVE_FIGURES:fig.savefig(FIG/f'LOSO_MAE_scales_{target}.png',dpi=175,bbox_inches='tight')
    plt.show();plt.close(fig)
print('All tables:',OUT)
print('Figures:',FIG)


## 06 — High-cover sensitivity and global LOSO skill (no aggregation rerun)

Targeted **histogram gradient boosting** with unweighted and high-cover-weighted training, on cached 75/100/125 m sample tables. All four forests are held out in turn. This block is **standalone after Sections 00–01** (it loads cached Parquet data and does not need Section 04 to be run again). Each fit is checkpointed to Drive; rerunning skips completed folds.

Report pooled LOSO R², site-equal macro MAE, a *site-balanced* pooled R², **within-site-centred R²**, MAE skill against the corresponding training-site mean baseline, and high-cover (observed >40%) bias/recall. Figures include observed-versus-predicted scatter by site, MOS tail calibration, and overall skill comparison. **Research caveat:** model ranking on these four LOSO sites is exploratory; the same held-out sites should not later be presented as an untouched final test set.

In [ ]:
# BLOCK 06 — uses existing cached samples and archived LOSO fold predictions; no UAV reprocessing
from pathlib import Path
from IPython.display import display
import time, gc
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
assert B06_RUN
B06_DIR=OUT/'06_highcover_LOSO';B06_FOLDS=B06_DIR/'folds';B06_FIG=B06_DIR/'figures'
for d in (B06_DIR,B06_FOLDS,B06_FIG):d.mkdir(parents=True,exist_ok=True)
def b06_metrics(df):
    y=df.observed.to_numpy(float);p=df.predicted.to_numpy(float)
    out=dict(n=len(df),R2=r2_score(y,p),MAE_pp=100*np.mean(np.abs(p-y)),
             RMSE_pp=100*np.sqrt(np.mean((p-y)**2)),bias_pp=100*np.mean(p-y))
    high=y>=B06_HIGH_THRESHOLD
    out['n_high']=int(high.sum())
    out['high_bias_pp']=100*np.mean(p[high]-y[high]) if high.any() else np.nan
    out['high_MAE_pp']=100*np.mean(np.abs(p[high]-y[high])) if high.any() else np.nan
    out['high_recall']=float(np.mean(p[high]>=B06_HIGH_THRESHOLD)) if high.any() else np.nan
    return out
def b06_global(df, baseline=None):
    # Equal-site pooled scores: each site contributes total weight 1/number_of_sites.
    sz=df.groupby('site').size();weight=df.site.map(lambda s:1/sz[s]).to_numpy(float)
    weight/=weight.sum();y=df.observed.to_numpy(float);p=df.predicted.to_numpy(float)
    mse=float(np.sum(weight*(p-y)**2));mae=float(np.sum(weight*np.abs(p-y)))
    mu=float(np.sum(weight*y));sst=float(np.sum(weight*(y-mu)**2))
    # Center *both* series on the observed site mean to suppress between-site variation.
    means=df.groupby('site')['observed'].mean()
    yc=y-df.site.map(means).to_numpy(float)
    pc=p-df.site.map(means).to_numpy(float)
    within_sst=float(np.sum(weight*yc**2))
    val=dict(pooled_R2=r2_score(y,p),balanced_pooled_R2=1-mse/sst if sst>0 else np.nan,
             within_site_R2=1-mse/within_sst if within_sst>0 else np.nan,
             macro_MAE_pp=100*mae,macro_RMSE_pp=100*np.sqrt(mse),
             macro_bias_pp=100*np.sum(weight*(p-y)))
    vals=[b06_metrics(g) for _,g in df.groupby('site')]
    val['macro_high_bias_pp']=float(np.nanmean([v['high_bias_pp'] for v in vals])) if any(v['n_high'] for v in vals) else np.nan
    val['macro_high_recall']=float(np.nanmean([v['high_recall'] for v in vals])) if any(v['n_high'] for v in vals) else np.nan
    if baseline is not None:
        base=b06_global(baseline)
        val['MAE_skill_vs_train_mean']=1-val['macro_MAE_pp']/base['macro_MAE_pp']
    return val
b06_records=[];b06_start=time.perf_counter()
for scale in B06_SCALES_M:
    sample_path=SAMPLE/f'cells_{scale}m.parquet'
    assert sample_path.is_file(),f'Run Section 03 once to produce {sample_path}'
    frame=pd.read_parquet(sample_path)
    print(f'\\nBLOCK 06 | {scale} m | {len(frame):,} cached cells',flush=True)
    for feature_set in B06_FEATURE_SETS:
        cols=[c for c in frame if any(c.startswith(f+'__') for f in FAMILY_NAMES[feature_set])]
        for target in B06_TARGETS:
            for holdout in SITE_MOSAICS:
                tr=frame[frame.site!=holdout];te=frame[frame.site==holdout]
                Xtr=tr[cols].to_numpy('float32');Xte=te[cols].to_numpy('float32')
                ytr=tr[target].to_numpy('float32');yte=te[target].to_numpy('float32')
                for strength in B06_WEIGHT_STRENGTHS:
                    tag=f'{scale}m_{feature_set}_{target}_{holdout}_w{strength}'
                    dest=B06_FOLDS/f'{tag}.csv'
                    if dest.exists() and not B06_REBUILD:
                        fold=pd.read_csv(dest)
                        print('CACHE',tag,flush=True)
                    else:
                        t0=time.perf_counter()
                        # Smooth upweighting begins at 20%, saturates at 60%;
                        # e.g. strength=5 yields weights from 1 to 6.
                        weights=1+strength*np.clip((ytr-.20)/.40,0,1)
                        model=HistGradientBoostingRegressor(learning_rate=.06,max_iter=200,
                            max_leaf_nodes=20,min_samples_leaf=40,l2_regularization=2,
                            early_stopping=False,random_state=SEED)
                        model.fit(Xtr,ytr,sample_weight=weights)
                        pred=np.clip(model.predict(Xte),0,1)
                        fold=te[['site','mosaic','row','col']].copy()
                        fold['observed']=yte;fold['predicted']=pred
                        fold.to_csv(dest,index=False)
                        print(f'DONE {tag} | MAE={100*np.mean(abs(pred-yte)):.2f} pp | R²={r2_score(yte,pred):.3f} | {time.perf_counter()-t0:.1f}s',flush=True)
                    b06_records.append(dict(scale_m=scale,feature_set=feature_set,target=target,
                                            holdout=holdout,strength=strength,**b06_metrics(fold)))
    del frame;gc.collect()
pd.DataFrame(b06_records).to_csv(B06_DIR/'06_per_site_metrics.csv',index=False)
global_records=[]
for scale in B06_SCALES_M:
 for feature_set in B06_FEATURE_SETS:
  for target in B06_TARGETS:
   baseline_parts=[]
   for site in SITE_MOSAICS:
    bpath=FOLDS/f'{scale}m_{feature_set}_{target}_{site}_TRAIN_MEAN.csv'
    if bpath.exists():baseline_parts.append(pd.read_csv(bpath)[['site','observed','predicted']])
    else:
     f=pd.read_parquet(SAMPLE/f'cells_{scale}m.parquet')
     tr=f[f.site!=site];te=f[f.site==site]
     baseline_parts.append(pd.DataFrame(dict(site=site,observed=te[target].to_numpy(),
                                          predicted=np.repeat(tr[target].mean(),len(te)))))
   baseline=pd.concat(baseline_parts,ignore_index=True)
   for strength in B06_WEIGHT_STRENGTHS:
    ds=[pd.read_csv(B06_FOLDS/f'{scale}m_{feature_set}_{target}_{site}_w{strength}.csv') for site in SITE_MOSAICS]
    df=pd.concat(ds,ignore_index=True)
    global_records.append(dict(scale_m=scale,feature_set=feature_set,target=target,strength=strength,
                               **b06_global(df,baseline)))
global_table=pd.DataFrame(global_records).sort_values(['target','macro_MAE_pp'])
global_table.to_csv(B06_DIR/'06_global_LOSO_metrics.csv',index=False)
display(global_table.round(3))
for target in B06_TARGETS:
 group=global_table[global_table.target==target]
 winner=group.sort_values('macro_MAE_pp').iloc[0]
 scale=int(winner.scale_m);family=winner.feature_set;strength=int(winner.strength)
 ds=[pd.read_csv(B06_FOLDS/f'{scale}m_{family}_{target}_{site}_w{strength}.csv') for site in SITE_MOSAICS]
 best=pd.concat(ds,ignore_index=True)
 fig,axes=plt.subplots(2,2,figsize=(10,9),constrained_layout=True)
 for ax,(site,g) in zip(axes.flat,best.groupby('site',sort=False)):
    ax.hexbin(g.observed*100,g.predicted*100,gridsize=38,mincnt=1,bins='log',cmap='viridis')
    ax.plot([0,100],[0,100],'--',color='gray',lw=1)
    mm=b06_metrics(g);ax.set(xlim=(0,100),ylim=(0,100),xlabel='UAV target (%)',
        ylabel='Satellite predicted (%)',title=f'{site}: R² {mm["R2"]:.2f}, MAE {mm["MAE_pp"]:.1f} pp')
 fig.suptitle(f'Weighted boosting LOSO | {target} | {scale}m {family} weight={strength}')
 fig.savefig(B06_FIG/f'06_best_LOSO_scatter_{target}.png',dpi=175)
 plt.show();plt.close(fig)
 # MOS calibration: observed abundance bins; inspect high-cover saturation explicitly.
 mos=best[best.site=='MOS'].copy()
 mos['bin']=pd.cut(mos.observed,bins=[0,.1,.2,.3,.4,.5,.6,.7,.8,1.0001],include_lowest=True)
 cal=mos.groupby('bin',observed=True).agg(n=('observed','size'),
        observed=('observed','mean'),predicted=('predicted','mean')).reset_index()
 cal.to_csv(B06_DIR/f'06_MOS_calibration_{target}.csv',index=False)
 fig,ax=plt.subplots(figsize=(7,5))
 ax.plot([0,100],[0,100],'--',color='gray',lw=1)
 ax.plot(cal.observed*100,cal.predicted*100,'o-',label='MOS binned mean')
 for _,rr in cal.iterrows():ax.annotate(str(int(rr['n'])),(100*rr.observed,100*rr.predicted),xytext=(4,4),textcoords='offset points',fontsize=7)
 ax.set(xlim=(0,100),ylim=(0,100),xlabel='Observed MOS fraction (%)',ylabel='Predicted MOS fraction (%)',
        title=f'MOS tail calibration: {target} | {scale}m {family} w={strength}')
 ax.legend();ax.grid(alpha=.2);fig.tight_layout()
 fig.savefig(B06_FIG/f'06_MOS_tail_{target}.png',dpi=175)
 plt.show();plt.close(fig)
 print(f'BEST {target}: {scale}m {family} weight={strength} | macro MAE={winner.macro_MAE_pp:.2f}pp | pooled R²={winner.pooled_R2:.3f} | within-site R²={winner.within_site_R2:.3f}',flush=True)
print('BLOCK 06 saved:',B06_DIR,'| elapsed min',round((time.perf_counter()-b06_start)/60,1),flush=True)


## 07 — ROI-restricted seasonal LOSO: reuse aggregates, build samples, fit folds

**Run requirements:** Section 00 (settings), then the three code cells below. The setup is self-contained and reuses NB04 cached UAV aggregates, not original full-mosaic sample tables. **Do not rerun Sections 02–06.** Uses the corrected NB04A `04A_quarterly_3year_gridfix` products. Five-mosaic and strict four-forest validation are separate; ROI coverage ≥90%.

In [ ]:
# 01 — Dependencies, input manifest, reuse cache
%pip -q install rasterio geopandas pyarrow scikit-learn matplotlib
import json,math,gc,time,hashlib
import numpy as np,pandas as pd,geopandas as gpd,rasterio
from rasterio.features import rasterize
from rasterio.warp import reproject,Resampling
from affine import Affine
from shapely.geometry import box
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score
from IPython.display import display
import matplotlib.pyplot as plt
from google.colab import drive
if not Path("/content/drive/MyDrive").exists(): drive.mount("/content/drive")
V3=FINAL/"Outputs/V3/04_satellite_Musanga/features"
OLD=FINAL/"Outputs/V3_1/04_satellite_full_warmstart"
AGG=OLD/"cache/aggregates"
QUART=FINAL/"Outputs/V3_1/04A_quarterly_3year_gridfix"
OUT=FINAL/"Outputs/V3_1/04B_roi_quarterly_LOSO"
SAMPLES=OUT/"samples";FOLDS=OUT/"folds";FIG=OUT/"figures"
for p in [SAMPLES,FOLDS,FIG]:p.mkdir(parents=True,exist_ok=True)
YEARS={"Baego":2023,"MOS":2024,"Yangambi":2024,"Yoko":2020}
WINDOWS={"Baego":(2021,2023),"MOS":(2022,2024),"Yangambi":(2022,2024),"Yoko":(2018,2020)}
MOSAICS={"Baego":["Baego.tif"],"MOS":["Mos_1_2_transparent_mosaic_group1_georef.tif",
"Mos_5_6_transparent_mosaic_group1_georef.tif"],"Yangambi":["Yangambi_georef.tif"],"Yoko":["Yoko_kvo_georef.tif"]}
GROUPS={"Baego":"Baego","Mos_1_2_transparent_mosaic_group1_georef":"MOS1_2",
"Mos_5_6_transparent_mosaic_group1_georef":"MOS5_6","Yangambi_georef":"Yangambi","Yoko_kvo_georef":"Yoko"}
manifest=FINAL/"Outputs/V3/00_input_audit/00_input_manifest.json"
if not ROI_PATH.exists() and manifest.exists():ROI_PATH=Path(json.loads(manifest.read_text())["roi"])
assert ROI_PATH.exists(),ROI_PATH
ROI=gpd.read_file(ROI_PATH)
assert ROI.crs is not None and len(ROI)>0,"ROI CRS or geometries missing"
print("ROI",ROI_PATH,"polygons",len(ROI),flush=True)
def path_feature(site,family):
    if family=="S2_QUARTERLY_3YR":
        a,b=WINDOWS[site];return QUART/f"{site}_S2_QUARTERLY_3YR_{a}_{b}_10m.tif"
    return V3/family/f"{site}_{family}_{YEARS[site]}_10m.tif"
def agg_path(mosaic,scale):return AGG/f"{Path(mosaic).stem}_{scale}m.tif"
for site,names in MOSAICS.items():
    for family in sorted(set(sum(FEATURE_SETS.values(),[]))):
        p=path_feature(site,family)
        assert p.exists(),f"Missing feature: {p}. Finish NB04A and QA before NB04B."
    for name in names:
        for scale in SCALES:
            assert agg_path(name,scale).exists(),f"Missing cached UAV aggregate: {agg_path(name,scale)}"
print("ALL source products found. Existing UAV aggregates will NOT be rebuilt.",flush=True)


In [ ]:
# 02 — Geometry grids, fractional ROI cover and cached sampling
def grid_for(site,scale):
    with rasterio.open(path_feature(site,"RAW_ANNUAL")) as ds:
        assert ds.crs is not None and abs(ds.transform.a-10)<.001 and abs(ds.transform.e+10)<.001
        f=scale/10
        return dict(height=math.ceil(ds.height/f),width=math.ceil(ds.width/f),
                    transform=ds.transform*Affine.scale(f,f),crs=ds.crs)
def warp_band(ds,index,g,mode=Resampling.average):
    a=np.full((g["height"],g["width"]),-9999.,dtype=np.float32)
    reproject(source=rasterio.band(ds,index),destination=a,src_nodata=ds.nodata,
              dst_transform=g["transform"],dst_crs=g["crs"],dst_nodata=-9999.,
              resampling=mode,num_threads=2)
    a[a==-9999]=np.nan
    return a
def roi_fraction(g):
    k=ROI_SUBPIXELS
    local=ROI.to_crs(g["crs"])
    fine=rasterize([(geom,1) for geom in local.geometry if geom is not None and not geom.is_empty],
           out_shape=(g["height"]*k,g["width"]*k),
           transform=g["transform"]*Affine.scale(1/k,1/k),fill=0,all_touched=False,dtype="uint8")
    return fine.reshape(g["height"],k,g["width"],k).mean(axis=(1,3))
def cube_on_grid(site,g):
    cubes=[];cols=[];quarter_clear=[]
    for family in ["RAW_ANNUAL","EMB_ANNUAL","S2_QUARTERLY_3YR"]:
        with rasterio.open(path_feature(site,family)) as ds:
            count_expected=44 if family=="S2_QUARTERLY_3YR" else None
            if count_expected:assert ds.count==count_expected,(site,ds.count)
            assert ds.crs==g["crs"],f"CRS mismatch {site} {family}"
            for b in range(1,ds.count+1):
                mode=Resampling.nearest if family=="S2_QUARTERLY_3YR" and b%11==0 else Resampling.average
                band=warp_band(ds,b,g,mode=mode)
                if family=="S2_QUARTERLY_3YR" and b%11==0:quarter_clear.append(band)
                cubes.append(band);cols.append(f"{family}__{b:02d}")
    return np.stack(cubes),cols,np.stack(quarter_clear)
SAMPLE_FRAMES={}
for scale in SCALES:
    cache=SAMPLES/f"roi_cells_{scale}m.parquet"
    if cache.exists() and not FORCE_SAMPLES:
        df=pd.read_parquet(cache)
        SAMPLE_FRAMES[scale]=df;print("ROI SAMPLE CACHE",scale,len(df),flush=True);continue
    rows=[]
    for site,names in MOSAICS.items():
        g=grid_for(site,scale)
        fraction=roi_fraction(g)
        cube,cols,clear=cube_on_grid(site,g)
        # All four quarterly clear count bands must satisfy minimum count.
        feature_valid=np.isfinite(cube).all(axis=0)&np.isfinite(clear).all(axis=0)&(clear>=MIN_QUARTER_CLEAR).all(axis=0)
        for name in names:
            with rasterio.open(agg_path(name,scale)) as ds:
                assert ds.crs==g["crs"] and ds.transform.almost_equals(g["transform"],precision=1e-6)
                y1,y2,uav_valid=ds.read()
            ok=(fraction>=MIN_ROI_COVERAGE)&(uav_valid>=MIN_UAV_COVERAGE)&feature_valid
            ok &= np.isfinite(y1)&np.isfinite(y2)&(y1>=0)&(y1<=1)&(y2>=0)&(y2<=1)
            rr,cc=np.where(ok)
            rng=np.random.default_rng(SEED+sum(map(ord,name))+scale)
            take=rng.choice(len(rr),min(len(rr),MAX_CELLS_PER_MOSAIC),replace=False)
            rr,cc=rr[take],cc[take]
            df=pd.DataFrame(cube[:,rr,cc].T,columns=cols)
            df.insert(0,"site",site);df.insert(1,"group",GROUPS[Path(name).stem])
            df.insert(2,"mosaic",name);df.insert(3,"row",rr);df.insert(4,"col",cc)
            df["roi_fraction"]=fraction[rr,cc]
            df["mean_probability"]=y1[rr,cc];df["cover_fraction_p05"]=y2[rr,cc]
            rows.append(df)
            print("ROI SAMPLES",scale,site,GROUPS[Path(name).stem],len(df),"candidate",len(np.where(ok)[0]),flush=True)
        del cube,clear;gc.collect()
    frame=pd.concat(rows,ignore_index=True)
    assert frame.group.nunique()==5,f"Only {frame.group.nunique()} groups with sampled cells"
    frame.to_parquet(cache,index=False);SAMPLE_FRAMES[scale]=frame
    print("SAVED",cache,len(frame),flush=True)


In [ ]:
# 03 — Five-mosaic versus four-forest LOSO; checkpoint each fold
def scores(y,p):
    y=np.asarray(y,float);p=np.asarray(p,float)
    high=y>=HIGH_COVER_THRESHOLD
    return dict(n=len(y),R2=float(r2_score(y,p)) if len(y)>1 else np.nan,
                MAE_pp=float(100*np.mean(abs(p-y))),RMSE_pp=float(100*np.sqrt(np.mean((p-y)**2))),
                bias_pp=float(100*np.mean(p-y)),n_high=int(high.sum()),
                high_bias_pp=float(100*np.mean((p-y)[high])) if high.any() else np.nan,
                high_recall=float(np.mean(p[high]>=HIGH_COVER_THRESHOLD)) if high.any() else np.nan)
records=[]
for scale,df in SAMPLE_FRAMES.items():
    for scheme in CV_SCHEMES:
        fold_col="group" if scheme=="FIVE_MOSAICS" else "site"
        for holdout in sorted(df[fold_col].unique()):
            train=df[df[fold_col]!=holdout];test=df[df[fold_col]==holdout]
            if len(train)<100 or len(test)<10:
                print("SKIP insufficient samples",scale,scheme,holdout,len(train),len(test),flush=True)
                continue
            for name,families in FEATURE_SETS.items():
                cols=[col for col in df.columns if any(col.startswith(f+"__") for f in families)]
                xtr=train[cols].to_numpy(np.float32);xte=test[cols].to_numpy(np.float32)
                assert np.isfinite(xtr).all() and np.isfinite(xte).all()
                for target in TARGETS:
                    ytr=train[target].to_numpy(np.float32);yte=test[target].to_numpy(np.float32)
                    for strength in WEIGHT_STRENGTHS:
                        key=f"{scheme}_{scale}m_{holdout}_{name}_{target}_w{strength}"
                        dest=FOLDS/f"{key}.parquet"
                        if dest.exists() and not FORCE_FOLDS:
                            fold=pd.read_parquet(dest);print("CACHE",key,flush=True)
                        else:
                            start=time.perf_counter()
                            weights=1+strength*np.clip((ytr-.20)/.40,0,1)
                            model=HistGradientBoostingRegressor(learning_rate=.06,max_iter=200,
                                max_leaf_nodes=20,min_samples_leaf=40,l2_regularization=2,
                                early_stopping=False,random_state=SEED)
                            model.fit(xtr,ytr,sample_weight=weights)
                            pred=np.clip(model.predict(xte),0,1)
                            fold=test[["site","group","mosaic","row","col"]].copy()
                            fold["observed"]=yte;fold["predicted"]=pred
                            fold.to_parquet(dest,index=False)
                            print("DONE",key,"seconds",round(time.perf_counter()-start,1),flush=True)
                        assert len(fold)==len(test),(key,"stale cache")
                        rec=dict(scheme=scheme,scale_m=scale,holdout=holdout,
                            predictors=name,target=target,weight=strength,**scores(fold.observed,fold.predicted))
                        records.append(rec)
        pd.DataFrame(records).to_csv(OUT/"fold_metrics_progress.csv",index=False)
results=pd.DataFrame(records);results.to_csv(OUT/"fold_metrics.csv",index=False)
display(results.round(3))


## 08 — Seasonal LOSO evaluation and MOS high-cover diagnostics

Compute pooled R², balanced R², within-group R², MAE, held-out MOS sensitivity. Separately distinguish mosaics versus forests. Outputs saved in `Final/Outputs/V3_1/04B_roi_quarterly_LOSO/` for provenance, despite the experiment being integrated in NB04.

In [ ]:
# 04 — Pooled and equal-group results, MOS tail, separate CV domains
summary=[]
for (scheme,scale,predictors,target,weight),g in results.groupby(
    ["scheme","scale_m","predictors","target","weight"]):
    frames=[]
    for heldout in g.holdout:
        p=FOLDS/f"{scheme}_{scale}m_{heldout}_{predictors}_{target}_w{weight}.parquet"
        frames.append(pd.read_parquet(p))
    d=pd.concat(frames,ignore_index=True)
    y=d.observed.to_numpy(float);pred=d.predicted.to_numpy(float)
    n=d.groupby("group" if scheme=="FIVE_MOSAICS" else "site").size()
    grouping="group" if scheme=="FIVE_MOSAICS" else "site"
    w=d[grouping].map(lambda s:1/n.loc[s]).to_numpy(float);w/=w.sum()
    ybar=np.dot(w,y);sst=np.dot(w,(y-ybar)**2);mse=np.dot(w,(y-pred)**2)
    grouped_means=d.groupby(grouping)["observed"].mean()
    site_center=y-d[grouping].map(grouped_means).to_numpy(float)
    within=np.dot(w,site_center**2)
    mos=d[d.site=="MOS"];hi=mos.observed>=HIGH_COVER_THRESHOLD
    summary.append(dict(scheme=scheme,scale_m=scale,predictors=predictors,target=target,
        weight=weight,n=len(d),pooled_R2=r2_score(y,pred),
        balanced_R2=1-mse/sst if sst>0 else np.nan,
        within_group_R2=1-mse/within if within>0 else np.nan,
        macro_MAE_pp=100*np.dot(w,abs(y-pred)),
        MOS_high_n=int(hi.sum()),
        MOS_high_bias_pp=100*(mos.predicted[hi]-mos.observed[hi]).mean() if hi.any() else np.nan,
        MOS_high_recall=float((mos.predicted[hi]>=HIGH_COVER_THRESHOLD).mean()) if hi.any() else np.nan))
summary=pd.DataFrame(summary)
summary.to_csv(OUT/"global_metrics.csv",index=False)
for scheme in CV_SCHEMES:
    print("\n",scheme,flush=True)
    display(summary[summary.scheme==scheme].sort_values("macro_MAE_pp").round(3))
print("Output",OUT,flush=True)


In [ ]:
# 05 — MOS high-cover scatterplots, held-out predictions only
if SAVE_PLOTS:
    for scheme in CV_SCHEMES:
        subset=summary[(summary.scheme==scheme)&(summary.target=="mean_probability")].sort_values("macro_MAE_pp")
        if not len(subset):continue
        winner=subset.iloc[0]
        scale=int(winner.scale_m);name=winner.predictors;wt=int(winner.weight)
        fig,axes=plt.subplots(1,2,figsize=(11,5),constrained_layout=True)
        for ax,group in zip(axes,["MOS1_2","MOS5_6"]):
            heldout=group if scheme=="FIVE_MOSAICS" else "MOS"
            f=FOLDS/f"{scheme}_{scale}m_{heldout}_{name}_mean_probability_w{wt}.parquet"
            if not f.exists():continue
            d=pd.read_parquet(f);d=d[d.group==group]
            ax.scatter(d.observed*100,d.predicted*100,s=7,alpha=.25,rasterized=True)
            ax.plot([0,100],[0,100],"--",color="grey")
            ax.axvline(HIGH_COVER_THRESHOLD*100,linestyle=":",color="grey")
            ax.set(xlim=(0,100),ylim=(0,100),title=f"{scheme}: {group} (n={len(d)})",
                   xlabel="UAV-derived mean probability (%)",ylabel="Satellite predicted (%)")
        fig.suptitle(f"{name} | {scale} m | weighting {wt}")
        fig.savefig(FIG/f"MOS_{scheme}_best.png",dpi=175)
        plt.show();plt.close(fig)


## Scientific caveats

- Five-mosaic LOSO can train on the *other* MOS mosaic. Four-forest LOSO holds both MOS mosaics out together and is the defensible strict geographic transfer test.
- Cached UAV aggregates were computed on full image grids; this notebook restricts their *sampled cells* to ROI with supersampled fractional coverage.
- Quarterly composites may include satellite observations after UAV flights. Temporal causality is not established.
- High-cover MOS diagnostics use the UAV model's pseudo-labels, not field-measured abundance.
- Comparisons use common complete-case features, which can reduce sample size; inspect per-mosaic sample counts.

## 09 — Cached LOSO diagnostics: by mosaic, MOS calibration, abundance support

Run after the new Block 07 (or after Block 07 setup and cached products exist). **No model fitting or raster exports.** This block reads only `04B_roi_quarterly_LOSO/samples` and `folds` and writes CSV and figures to `diagnostics/`. Primary interpretation is **FOUR_FORESTS**. High-cover thresholds refer to the UAV-derived mean probability, not independent ground truth.

In [ ]:
# BLOCK 09 — CACHED-DATA DIAGNOSTICS, NO FITTING
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import r2_score
from matplotlib.lines import Line2D
D09_BASE=Path("/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/04B_roi_quarterly_LOSO")
D09_FOLDS=D09_BASE/"folds"
D09_SAMPLES=D09_BASE/"samples"
D09_OUT=D09_BASE/"diagnostics"
D09_OUT.mkdir(parents=True,exist_ok=True)
D09_SCALES=[75,100,125]
D09_COMPARISONS=[("EMB",0),("QUARTERLY",2),("EMB_QUARTERLY",2),
                 ("EMB_ANNUAL_QUARTERLY",2)]
D09_BINS=np.array([0,.05,.10,.15,.20,.30,.40,.50,.60,.75,1.000001])
D09_MIN_BIN_N=5
D09_THRESHOLD=.40
D09_SCHEME="FOUR_FORESTS"
def d09_fold(scale,holdout,features,weight,scheme=D09_SCHEME):
    f=D09_FOLDS/f"{scheme}_{scale}m_{holdout}_{features}_mean_probability_w{weight}.parquet"
    if not f.is_file(): raise FileNotFoundError(f"Missing cached fold: {f}; run Block 07.")
    d=pd.read_parquet(f)
    required={"site","group","mosaic","observed","predicted"}
    assert required.issubset(d.columns), (f,required-set(d.columns))
    d=d.copy(); d["holdout"]=holdout;d["features"]=features;d["weight"]=weight
    return d
def d09_stats(d):
    y=d["observed"].to_numpy(float);p=d["predicted"].to_numpy(float)
    good=np.isfinite(y)&np.isfinite(p);y=y[good];p=p[good]
    h=y>=D09_THRESHOLD
    return dict(n=len(y),R2=r2_score(y,p) if len(y)>1 and np.var(y)>0 else np.nan,
                MAE_pp=100*np.mean(abs(p-y)),RMSE_pp=100*np.sqrt(np.mean((p-y)**2)),
                bias_pp=100*np.mean(p-y),n_high=int(h.sum()),
                high_bias_pp=100*np.mean((p-y)[h]) if h.any() else np.nan,
                high_recall=float(np.mean(p[h]>=D09_THRESHOLD)) if h.any() else np.nan,
                observed_p90_pp=100*np.quantile(y,.9),
                observed_max_pp=100*np.max(y))
# Build per-mosaic results from strict forest holdout (both MOS mosaics held out together)
rows=[]
for scale in D09_SCALES:
    for features,weight in D09_COMPARISONS:
        for heldout in ["Baego","MOS","Yangambi","Yoko"]:
            df=d09_fold(scale,heldout,features,weight)
            for group,part in df.groupby("group"):
                rows.append(dict(scale_m=scale,forest_holdout=heldout,mosaic_group=group,
                    predictors=features,weight=weight,**d09_stats(part)))
per_mosaic=pd.DataFrame(rows)
per_mosaic.to_csv(D09_OUT/"09_per_mosaic_strict_LOSO.csv",index=False)
print("STRICT FOUR-FOREST: performance by individual mosaic (high threshold=40%)")
display(per_mosaic[per_mosaic.scale_m==75].round(3))
# MOS calibration by observed-abundance bin
calibration=[]
for scale in D09_SCALES:
    for features,weight in D09_COMPARISONS:
        df=d09_fold(scale,"MOS",features,weight)
        for group,part in df.groupby("group"):
            idx=np.digitize(part["observed"].to_numpy(float),D09_BINS,right=False)-1
            for i in range(len(D09_BINS)-1):
                sel=part.iloc[np.where(idx==i)[0]]
                if len(sel)==0: continue
                calibration.append(dict(scale_m=scale,mosaic_group=group,
                    predictors=features,weight=weight,
                    bin_lower=D09_BINS[i],bin_upper=min(1,D09_BINS[i+1]),
                    n=len(sel),observed_mean_pp=100*sel["observed"].mean(),
                    predicted_mean_pp=100*sel["predicted"].mean(),
                    mean_bias_pp=100*(sel["predicted"]-sel["observed"]).mean(),
                    predicted_ge40_fraction=float((sel["predicted"]>=D09_THRESHOLD).mean())))
cal=pd.DataFrame(calibration)
cal.to_csv(D09_OUT/"09_MOS_calibration_bins.csv",index=False)
print("MOS calibration, bins containing >=5 cells, 75 m:")
display(cal[(cal.scale_m==75)&(cal.n>=D09_MIN_BIN_N)].round(2))
# Critical training-support diagnostic: only the OTHER forests are training for strict MOS holdout.
support=[]
for scale in D09_SCALES:
    p=D09_SAMPLES/f"roi_cells_{scale}m.parquet"
    if not p.is_file(): raise FileNotFoundError(f"Missing cached ROI sample: {p}")
    df=pd.read_parquet(p)
    for label,part in [("TRAIN_excluding_MOS",df[df["site"]!="MOS"]),
                       ("TEST_MOS1_2",df[df["group"]=="MOS1_2"]),
                       ("TEST_MOS5_6",df[df["group"]=="MOS5_6"])]:
        y=part["mean_probability"].to_numpy(float)
        support.append(dict(scale_m=scale,population=label,n=len(y),
            mean_pp=100*np.mean(y),p90_pp=100*np.quantile(y,.9),
            p95_pp=100*np.quantile(y,.95),max_pp=100*np.max(y),
            pct_above20=100*np.mean(y>=.20),
            pct_above40=100*np.mean(y>=.40),
            pct_above60=100*np.mean(y>=.60)))
sup=pd.DataFrame(support)
sup.to_csv(D09_OUT/"09_MOS_train_test_abundance_support.csv",index=False)
print("Critical test: do Baego/Yangambi/Yoko contain enough high-cover training examples?")
display(sup.round(2))
# One figure per scale: per-MOS scatter, calibration, and abundance support
for scale in D09_SCALES:
    fig,axs=plt.subplots(2,2,figsize=(12,10),constrained_layout=True)
    for ax,group in zip(axs[0],["MOS1_2","MOS5_6"]):
        for features,weight in D09_COMPARISONS:
            df=d09_fold(scale,"MOS",features,weight)
            d=df[df["group"]==group]
            ax.scatter(100*d["observed"],100*d["predicted"],s=7,alpha=.22,
                       label=f"{features} w{weight}",rasterized=True)
        ax.plot([0,100],[0,100],linestyle="--",color="grey")
        ax.axvline(40,color="grey",linestyle=":")
        ax.set(xlim=(0,100),ylim=(0,100),title=f"{group} — strict forest holdout, {scale} m",
               xlabel="UAV-derived mean Musanga probability (%)",
               ylabel="Satellite-predicted mean probability (%)")
        ax.legend(fontsize=7)
    ax=axs[1,0]
    for features,weight in D09_COMPARISONS:
        z=cal[(cal.scale_m==scale)&(cal.n>=D09_MIN_BIN_N)&
              (cal.predictors==features)&(cal.weight==weight)]
        for group in ["MOS1_2","MOS5_6"]:
            zz=z[z.mosaic_group==group].sort_values("observed_mean_pp")
            ax.plot(zz.observed_mean_pp,zz.predicted_mean_pp,"-o",markersize=3,
                    label=f"{group} {features} w{weight}")
    ax.plot([0,100],[0,100],"--",color="grey")
    ax.set(xlim=(0,100),ylim=(0,100),xlabel="Mean UAV probability by observed bin (%)",
           ylabel="Mean satellite prediction (%)",title="MOS observed-bin calibration (n>=5)")
    ax.legend(fontsize=6,ncol=2)
    ax=axs[1,1]
    bins=np.linspace(0,100,26)
    for label,part in [("Train: other forests",pd.read_parquet(D09_SAMPLES/f"roi_cells_{scale}m.parquet").query("site != 'MOS'")),
                       ("Test: MOS1_2",pd.read_parquet(D09_SAMPLES/f"roi_cells_{scale}m.parquet").query("group == 'MOS1_2'")),
                       ("Test: MOS5_6",pd.read_parquet(D09_SAMPLES/f"roi_cells_{scale}m.parquet").query("group == 'MOS5_6'"))]:
        ax.hist(100*part["mean_probability"],bins=bins,histtype="step",density=True,
                linewidth=1.6,label=f"{label} (n={len(part)})")
    ax.axvline(40,color="grey",linestyle=":")
    ax.set(xlabel="UAV-derived mean Musanga probability (%)",
           ylabel="Sample density",title="Train/test abundance support")
    ax.legend(fontsize=8)
    fig.savefig(D09_OUT/f"09_MOS_strict_LOSO_diagnostics_{scale}m.png",dpi=175)
    plt.show();plt.close(fig)
print("Saved all Block 09 diagnostics to:",D09_OUT,flush=True)


## 10 — Abundance-balanced training sensitivity (cached ROI data, strict forest holdouts)

Run this cell alone after Block 07 has generated the cached ROI Parquet samples. **No UAV aggregation, satellite downloads, or previous benchmark reruns.** Compare a standard uniform-weight baseline with inverse-frequency bin weights of varying strengths. All weights are calculated **only from each fold's training forests**, capped to avoid overfitting the very few high-cover observations. The held-out forest retains its *natural, imbalanced distribution* for honest evaluation. Report 0–20%, 20–40%, ≥40% errors separately and overall metrics, including explicit MOS1–2 versus MOS5–6 diagnostics. **Do not create synthetic high-cover labels or mix MOS data into its strict holdout.**

In [ ]:
# BLOCK 10 — run standalone; all settings local at top
# === USER SETTINGS (Block 10) ===
B10_SCALES=[75,100,125]
B10_PREDICTORS={"EMB":["EMB_ANNUAL"],"EMB_QUARTERLY":["EMB_ANNUAL","S2_QUARTERLY_3YR"]}
B10_EDGES=[0,.10,.20,.30,.40,1.000001]
B10_BALANCE_POWERS=[0,.5,1.]  # 0=original, 0.5=sqrt inverse frequency, 1=full inverse frequency
B10_MAX_WEIGHT_RATIO=6.0
B10_SEED=20261022
B10_FORCE=False
# === END SETTINGS ===
from pathlib import Path
import numpy as np,pandas as pd,time
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score
from IPython.display import display
from google.colab import drive
if not Path("/content/drive/MyDrive").exists():drive.mount("/content/drive")
base=Path("/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/04B_roi_quarterly_LOSO")
samples=base/"samples"
folder=base/"10_abundance_balance"
fold_dir=folder/"folds";fold_dir.mkdir(parents=True,exist_ok=True)
forest_names=["Baego","MOS","Yangambi","Yoko"]
def balancing_weights(y,power):
    category=np.clip(np.digitize(y,B10_EDGES)-1,0,len(B10_EDGES)-2)
    counts=np.bincount(category,minlength=len(B10_EDGES)-1)
    nonempty=counts[counts>0]
    if not len(nonempty):raise ValueError("No training data")
    # Inverse frequency, capped relative to the most common bin's weight
    raw=np.where(counts>0,(counts.max()/np.maximum(counts,1))**power,0.)
    raw=np.minimum(raw,B10_MAX_WEIGHT_RATIO)
    weights=raw[category].astype("float64")
    weights/=np.mean(weights)
    return weights,counts
def b10_metrics(df):
    y=df["observed"].to_numpy(float);p=df["predicted"].to_numpy(float)
    out=dict(n=len(df),R2=r2_score(y,p) if len(y)>1 and np.var(y)>1e-12 else np.nan,
             MAE_pp=100*np.mean(abs(y-p)),RMSE_pp=100*np.sqrt(np.mean((y-p)**2)),
             bias_pp=100*np.mean(p-y))
    for name,mask in [("low",y<.20),("mid",(y>=.20)&(y<.40)),("high",y>=.40)]:
        out[f"n_{name}"]=int(mask.sum())
        out[f"MAE_{name}_pp"]=100*np.mean(abs(y[mask]-p[mask])) if mask.any() else np.nan
        out[f"bias_{name}_pp"]=100*np.mean((p-y)[mask]) if mask.any() else np.nan
    high=y>=.40
    out["high_recall"]=float(np.mean(p[high]>=.40)) if high.any() else np.nan
    return out
rows=[];bin_rows=[]
for scale in B10_SCALES:
    path=samples/f"roi_cells_{scale}m.parquet"
    if not path.exists():raise FileNotFoundError(f"Block 07 ROI sample cache missing: {path}")
    data=pd.read_parquet(path)
    assert set(forest_names)==set(data.site.unique()),data.site.unique()
    for heldout in forest_names:
        train=data[data.site!=heldout]
        test=data[data.site==heldout]
        ytr=train.mean_probability.to_numpy(np.float32)
        assert len(train)>50 and len(test)>10
        for name,families in B10_PREDICTORS.items():
            cols=[c for c in data.columns if any(c.startswith(f+"__") for f in families)]
            assert cols,f"No feature columns for {name}"
            xtr=train[cols].to_numpy(np.float32);xte=test[cols].to_numpy(np.float32)
            assert np.isfinite(xtr).all() and np.isfinite(xte).all(),f"Missing features: {name}"
            for power in B10_BALANCE_POWERS:
                label=f"FOUR_FORESTS_{scale}m_{heldout}_{name}_power{power:g}"
                saved=fold_dir/f"{label}.parquet"
                weights,counts=balancing_weights(ytr,power)
                for i,n in enumerate(counts):
                    bin_rows.append(dict(scale_m=scale,heldout=heldout,predictors=name,
                        balance_power=power,bin_lower=B10_EDGES[i],
                        bin_upper=min(1.,B10_EDGES[i+1]),n_train=int(n),
                        effective_weight=float(weights[np.clip(np.digitize(ytr,B10_EDGES)-1,0,len(counts)-1)==i].mean()) if n else np.nan))
                if saved.exists() and not B10_FORCE:
                    pred=pd.read_parquet(saved)
                    print("CACHE",label,flush=True)
                else:
                    t=time.perf_counter()
                    model=HistGradientBoostingRegressor(learning_rate=.06,max_iter=200,
                        max_leaf_nodes=20,min_samples_leaf=40,l2_regularization=2,
                        early_stopping=False,random_state=B10_SEED)
                    model.fit(xtr,ytr,sample_weight=weights)
                    pred=test[["site","group","mosaic","row","col"]].copy()
                    pred["observed"]=test["mean_probability"].to_numpy(np.float32)
                    pred["predicted"]=np.clip(model.predict(xte),0,1)
                    pred.to_parquet(saved,index=False)
                    print("FIT",label,"seconds",round(time.perf_counter()-t,1),flush=True)
                assert len(pred)==len(test),"Stale fold cache — set B10_FORCE=True"
                for group,subset in pred.groupby("group"):
                    rows.append(dict(scale_m=scale,heldout=heldout,mosaic_group=group,
                        predictors=name,balance_power=power,**b10_metrics(subset)))
    pd.DataFrame(rows).to_csv(folder/"10_per_mosaic_progress.csv",index=False)
metrics=pd.DataFrame(rows)
metrics.to_csv(folder/"10_per_mosaic_metrics.csv",index=False)
pd.DataFrame(bin_rows).to_csv(folder/"10_training_bin_counts_weights.csv",index=False)
print("MOS per-mosaic balanced-training comparison:")
display(metrics[metrics.heldout=="MOS"].round(3))
summary=[]
for (scale,name,power),part in metrics.groupby(["scale_m","predictors","balance_power"]):
    fold_parts=[]
    for forest in forest_names:
        d=pd.read_parquet(fold_dir/f"FOUR_FORESTS_{scale}m_{forest}_{name}_power{power:g}.parquet")
        fold_parts.append(d)
    pooled=pd.concat(fold_parts,ignore_index=True)
    sizes=pooled.groupby("site").size()
    w=pooled.site.map(lambda s:1/sizes[s]).to_numpy(float);w/=w.sum()
    y=pooled.observed.to_numpy(float);pred=pooled.predicted.to_numpy(float)
    mean=np.dot(w,y);sst=np.dot(w,(y-mean)**2);mse=np.dot(w,(y-pred)**2)
    group_mean=pooled.groupby("site")["observed"].mean()
    centered=y-pooled.site.map(group_mean).to_numpy(float)
    within=np.dot(w,centered**2)
    mos=pooled[pooled.site=="MOS"]
    hi=mos.observed>=.40
    summary.append(dict(scale_m=scale,predictors=name,balance_power=power,
        pooled_R2=r2_score(y,pred),balanced_R2=1-mse/sst if sst else np.nan,
        within_site_R2=1-mse/within if within else np.nan,
        macro_MAE_pp=100*np.dot(w,abs(y-pred)),
        MOS_high_n=int(hi.sum()),
        MOS_high_bias_pp=100*(mos.predicted[hi]-mos.observed[hi]).mean() if hi.any() else np.nan,
        MOS_high_recall=(mos.predicted[hi]>=.40).mean() if hi.any() else np.nan,
        MOS_low_MAE_pp=100*abs(mos.predicted[mos.observed<.20]-mos.observed[mos.observed<.20]).mean()))
summary=pd.DataFrame(summary).sort_values(["scale_m","predictors","balance_power"])
summary.to_csv(folder/"10_global_metrics.csv",index=False)
print("STRICT FOUR-FOREST LOSO: higher high-cover recall is meaningful only if macro MAE and low-cover error remain acceptable")
display(summary.round(3))
print("Outputs:",folder,flush=True)


## 11 — Spatially blocked MOS adaptation: 0%, 5%, 10%, 20%

**Question:** How much MOS UAV-labelled reference coverage is needed to improve satellite prediction of high Musanga abundance?

This experiment reuses Block 07 cached ROI samples and trains on Baego + Yangambi + Yoko, with nested additions of MOS training *spatial blocks*. It reserves a common MOS test area within each repeat, disjoint from the whole 20% adaptation pool, with a **200 m embargo around all candidate adaptation cells**. The fixed test set makes the four MOS training fractions directly comparable. MOS 1–2 and MOS 5–6 are reported separately. There are 3 spatial repeats. Percentages are approximate fractions of the **cached MOS cells**, not physical area, and realized fractions are reported explicitly.

**Run Block 11 directly.** No Blocks 01–10, raster extraction or UAV aggregation required. The test data are never used to select fitting weights or features. This is an *adapted/local transfer* experiment, not strict forest LOSO; baseline 0% uses the same spatially reserved evaluation cells to enable fair comparisons. More training cells do **not** establish independent field truth because targets remain UAV-model predictions.

In [ ]:
# BLOCK 11 — standalone spatial-blocked MOS adaptation (cached 75/100/125 m ROI samples)
# =============== SETTINGS (edit here only) ===============
B11_SCALES=[75,100,125]
B11_FRACTIONS=[0.00,0.05,0.10,0.20]
B11_FEATURES={
    "EMB":["EMB_ANNUAL"],
    "EMB_QUARTERLY":["EMB_ANNUAL","S2_QUARTERLY_3YR"]
}
B11_REPEATS=3
B11_BLOCK_SIZE_M=400.0
B11_BUFFER_M=200.0
B11_SEED=20261022
B11_FORCE=False
B11_HIGH_THRESHOLD=0.40
# =========================================================
from pathlib import Path
import numpy as np,pandas as pd,rasterio,time,math
from scipy.spatial import cKDTree
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score
from IPython.display import display
import matplotlib.pyplot as plt
from google.colab import drive
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
B11_FINAL=Path("/content/drive/MyDrive/Colab Notebooks/Musanga/Final")
B11_ROOT=B11_FINAL/"Outputs/V3_1/04B_roi_quarterly_LOSO"
B11_SAMPLE_DIR=B11_ROOT/"samples"
B11_DIR=B11_ROOT/"11_MOS_spatial_adaptation"
B11_FOLD_DIR=B11_DIR/"folds"
B11_FIG_DIR=B11_DIR/"figures"
for folder in (B11_DIR,B11_FOLD_DIR,B11_FIG_DIR):
    folder.mkdir(parents=True,exist_ok=True)
B11_ANNUAL=B11_FINAL/"Outputs/V3/04_satellite_Musanga/features/RAW_ANNUAL/MOS_RAW_ANNUAL_2024_10m.tif"
assert B11_ANNUAL.is_file(),f"Missing canonical MOS raster: {B11_ANNUAL}"
assert B11_FRACTIONS==sorted(B11_FRACTIONS) and B11_FRACTIONS[0]==0 and B11_FRACTIONS[-1]==0.20
assert B11_BUFFER_M>0 and B11_BLOCK_SIZE_M>=B11_BUFFER_M
with rasterio.open(B11_ANNUAL) as d:
    B11_TRANSFORM=d.transform
    B11_CRS=d.crs
assert B11_CRS.is_projected,"Spatial blocks require a projected CRS in metres"
def b11_metric(d):
    y=d["observed"].to_numpy(float);p=d["predicted"].to_numpy(float)
    h=y>=B11_HIGH_THRESHOLD
    low=y<.20
    out=dict(n=int(len(d)),
             R2=float(r2_score(y,p)) if len(y)>1 and np.var(y)>1e-12 else np.nan,
             MAE_pp=100*float(np.mean(abs(y-p))),
             RMSE_pp=100*float(np.sqrt(np.mean((y-p)**2))),
             bias_pp=100*float(np.mean(p-y)),
             n_high=int(h.sum()),
             high_bias_pp=100*float(np.mean((p-y)[h])) if h.any() else np.nan,
             high_recall=float(np.mean(p[h]>=B11_HIGH_THRESHOLD)) if h.any() else np.nan,
             low_MAE_pp=100*float(np.mean(abs((p-y)[low]))) if low.any() else np.nan)
    return out
def b11_select_blocks(mos,scale,repeat):
    # Physical coordinates: canonical MOS 10 m grid, with scaled cell centre.
    xf=B11_TRANSFORM.a*(scale/10.)
    yf=B11_TRANSFORM.e*(scale/10.)
    x=B11_TRANSFORM.c+(mos["col"].to_numpy(float)+.5)*xf
    y=B11_TRANSFORM.f+(mos["row"].to_numpy(float)+.5)*yf
    mos=mos.copy()
    mos["x_m"]=x;mos["y_m"]=y
    mos["block_x"]=np.floor(x/B11_BLOCK_SIZE_M).astype("int64")
    mos["block_y"]=np.floor(y/B11_BLOCK_SIZE_M).astype("int64")
    # Groups distinguish mosaic membership but the spatial embargo uses global XY.
    mos["block_id"]=(mos["group"].astype(str)+"__"+
                     mos["block_x"].astype(str)+"__"+mos["block_y"].astype(str))
    rng=np.random.default_rng(B11_SEED+scale*100+repeat)
    # Select ~20% of MOS cells into the candidate local-reference pool,
    # stratified independently by MOS1_2 and MOS5_6.
    ordered_blocks=[]
    for mosaic_group,sub in mos.groupby("group",sort=True):
        counts=sub.groupby("block_id").size()
        permutation=rng.permutation(len(counts))
        cumulative=0;goal=.20*len(sub)
        for i in permutation:
            block=counts.index[i]
            if cumulative>=goal:break
            ordered_blocks.append((mosaic_group,block,int(counts.iloc[i])))
            cumulative+=int(counts.iloc[i])
    pool_ids={b for _,b,_ in ordered_blocks}
    mos["in_pool"]=mos["block_id"].isin(pool_ids)
    pool=mos[mos["in_pool"]].copy()
    assert len(pool)>0 and len(pool)<len(mos),"Spatial split cannot be constructed"
    # Reserve a common holdout for all fractions, away from ANY pool cells.
    holdout=mos[~mos["in_pool"]].copy()
    distance=cKDTree(pool[["x_m","y_m"]].to_numpy(float)).query(
        holdout[["x_m","y_m"]].to_numpy(float),k=1)[0]
    holdout=holdout.loc[distance>=B11_BUFFER_M].copy()
    assert len(holdout)>=30,("Too few holdout cells after buffer",scale,repeat,len(holdout))
    assert set(holdout["group"])=={"MOS1_2","MOS5_6"},(
        "Both MOS mosaic test groups required",scale,repeat)
    # Nested acquisition blocks. Choose separately within each mosaic group;
    # increments are approximate percentages of ALL cached MOS cells.
    ids_by_fraction={}
    for fraction in B11_FRACTIONS:
        chosen=set()
        for group in ["MOS1_2","MOS5_6"]:
            order=[(block,n) for g,block,n in ordered_blocks if g==group]
            desired=fraction*int((mos["group"]==group).sum())
            selected=0
            for block,n in order:
                if selected>=desired: break
                chosen.add(block);selected+=n
        ids_by_fraction[fraction]=chosen
    assert all(ids_by_fraction[a].issubset(ids_by_fraction[b])
               for a,b in zip(B11_FRACTIONS[:-1],B11_FRACTIONS[1:]))
    return mos,holdout,ids_by_fraction
b11_metrics=[];b11_support=[];b11_splits=[]
for scale in B11_SCALES:
    path=B11_SAMPLE_DIR/f"roi_cells_{scale}m.parquet"
    if not path.is_file():raise FileNotFoundError(f"Missing Block 07 sample cache: {path}")
    data=pd.read_parquet(path)
    assert {"site","group","row","col","mean_probability"}.issubset(data.columns)
    other=data[data["site"]!="MOS"].copy()
    base_mos=data[data["site"]=="MOS"].copy()
    assert set(base_mos.group.unique())=={"MOS1_2","MOS5_6"}
    for repeat in range(B11_REPEATS):
        mos,test,ids=b11_select_blocks(base_mos,scale,repeat)
        # Evaluate the EXACT same MOS cells for all 4 adaptation fractions.
        test_key=set(zip(test["group"],test["row"],test["col"]))
        for fraction in B11_FRACTIONS:
            local=mos[mos["block_id"].isin(ids[fraction])]
            # Confirm physical embargo and absence of overlapping cells.
            if len(local):
                actual_gap=cKDTree(local[["x_m","y_m"]].to_numpy()).query(
                    test[["x_m","y_m"]].to_numpy(),k=1)[0].min()
                assert actual_gap>=B11_BUFFER_M-1e-6,(scale,repeat,fraction,actual_gap)
            local_key=set(zip(local["group"],local["row"],local["col"]))
            assert not(test_key&local_key),"Spatial split leakage"
            train=pd.concat([other,local],ignore_index=True)
            train_count=dict(scale_m=scale,repeat=repeat,fraction_requested=fraction,
                n_mos_total=len(mos),n_mos_pool=int(mos["in_pool"].sum()),
                n_mos_training=len(local),fraction_realized=len(local)/len(mos),
                n_mos_test=len(test),test_n_high=int((test.mean_probability>=B11_HIGH_THRESHOLD).sum()),
                n_other_forest_training=len(other))
            b11_splits.append(train_count)
            print("SPLIT",train_count,flush=True)
            ytr=train["mean_probability"].to_numpy("float32")
            for feature_name,families in B11_FEATURES.items():
                cols=[c for c in data.columns if any(c.startswith(f+"__") for f in families)]
                assert cols,"No predictors for "+feature_name
                Xtr=train[cols].to_numpy("float32")
                Xte=test[cols].to_numpy("float32")
                assert np.isfinite(Xtr).all() and np.isfinite(Xte).all(),(
                    "Missing spectral predictors",scale,feature_name)
                tag=f"{scale}m_r{repeat}_{feature_name}_local{int(round(100*fraction)):02d}"
                cached=B11_FOLD_DIR/f"{tag}.parquet"
                if cached.exists() and not B11_FORCE:
                    pred=pd.read_parquet(cached)
                    assert (len(pred)==len(test) and
                            np.allclose(pred["observed"].to_numpy(),test["mean_probability"].to_numpy(),
                                        atol=1e-6)),("Stale cached fold",cached)
                    print("CACHE",tag,flush=True)
                else:
                    t0=time.perf_counter()
                    model=HistGradientBoostingRegressor(
                        learning_rate=.06,max_iter=200,max_leaf_nodes=20,
                        min_samples_leaf=40,l2_regularization=2,
                        early_stopping=False,random_state=B11_SEED)
                    model.fit(Xtr,ytr)
                    pred=test[["site","group","mosaic","row","col"]].copy()
                    pred["observed"]=test["mean_probability"].to_numpy("float32")
                    pred["predicted"]=np.clip(model.predict(Xte),0,1)
                    pred.to_parquet(cached,index=False)
                    print("FIT",tag,"seconds",round(time.perf_counter()-t0,1),flush=True)
                for group,sub in pred.groupby("group"):
                    b11_metrics.append(dict(scale_m=scale,repeat=repeat,
                        fraction_requested=fraction,fraction_realized=len(local)/len(mos),
                        predictor_set=feature_name,mosaic_group=group,**b11_metric(sub)))
                b11_metrics.append(dict(scale_m=scale,repeat=repeat,
                    fraction_requested=fraction,fraction_realized=len(local)/len(mos),
                    predictor_set=feature_name,mosaic_group="MOS_ALL",**b11_metric(pred)))
    pd.DataFrame(b11_metrics).to_csv(B11_DIR/"11_metrics_progress.csv",index=False)
metrics=pd.DataFrame(b11_metrics)
metrics.to_csv(B11_DIR/"11_per_repeat_metrics.csv",index=False)
pd.DataFrame(b11_splits).to_csv(B11_DIR/"11_training_support.csv",index=False)
summary=metrics.groupby(["scale_m","fraction_requested","predictor_set","mosaic_group"],as_index=False).agg(
    n_test_mean=("n","mean"),fraction_realized_mean=("fraction_realized","mean"),
    R2_mean=("R2","mean"),R2_sd=("R2","std"),
    MAE_pp_mean=("MAE_pp","mean"),MAE_pp_sd=("MAE_pp","std"),
    bias_pp_mean=("bias_pp","mean"),
    high_n_mean=("n_high","mean"),high_recall_mean=("high_recall","mean"),
    high_bias_pp_mean=("high_bias_pp","mean"),low_MAE_pp_mean=("low_MAE_pp","mean"))
summary.to_csv(B11_DIR/"11_summary.csv",index=False)
print("\n=== MOS adaptation response: pooled mosaics, averages across spatial repeats ===")
display(summary[summary.mosaic_group=="MOS_ALL"].round(3))
print("\n=== MOS1_2 and MOS5_6 separately at 75 m ===")
display(summary[(summary.scale_m==75)&(summary.mosaic_group!="MOS_ALL")].round(3))
for scale in B11_SCALES:
    subset=summary[(summary.scale_m==scale)&(summary.mosaic_group=="MOS_ALL")]
    fig,axes=plt.subplots(1,3,figsize=(13,3.7),constrained_layout=True)
    for name,grp in subset.groupby("predictor_set"):
        grp=grp.sort_values("fraction_requested")
        x=100*grp["fraction_realized_mean"].to_numpy()
        for ax,col,title in zip(axes,["MAE_pp_mean","R2_mean","high_recall_mean"],
                                ["MOS MAE (pp)","MOS R²","MOS high-cover recall"]):
            ax.plot(x,grp[col],marker="o",label=name)
            ax.set(xlabel="Realized local MOS training (% of MOS cells)",title=title)
    axes[0].set_ylabel("Percentage points")
    axes[2].set_ylabel("Fraction of observed MOS cells ≥40% recovered")
    axes[0].legend(fontsize=8)
    fig.suptitle(f"Block 11: {scale} m, spatially blocked MOS adaptation")
    fig.savefig(B11_FIG_DIR/f"11_adaptation_{scale}m.png",dpi=175)
    plt.show();plt.close(fig)
print("Block 11 outputs:",B11_DIR,flush=True)


## 12 — Interpretable temporal features: seasonal S2 indices and strict geographic transfer

**Runs standalone from the cached Block 07 ROI samples.** Compare annual embeddings, 44-band raw quarterly S2, and engineered seasonal spectral features using the same strict FOUR_FORESTS LOSO. New models are limited to engineered temporal indices (alone and together with embeddings); baseline predictions are **reused from Block 07**, not refit. The indices are computed from the four pooled three-year quarterly median composites: this tests **repeating seasonal contrast**, not interannual dynamics, disturbance timing or phenology observed every year. An ecological relationship is *associational*, not evidence of causation or species-specific mechanism. Target is UAV-model-derived Musanga mean probability. Outputs include site-wise rank associations and low/high abundance seasonal spectral signatures.

In [ ]:
# BLOCK 12 — engineering temporal signatures from cached samples, NO satellite export
# ========= SETTINGS =========
T12_SCALES=[75,100,125]
T12_SEASONAL_INDICES=["NDVI","NDMI","NDRE","NBR","EVI2"]
T12_BASELINES=[("EMB",0),("QUARTERLY",0),("EMB_QUARTERLY",0),("EMB_QUARTERLY",2)]
T12_NEW_MODELS=["INDICES","EMB_INDICES"]
T12_SEED=20261022
T12_FORCE=False
T12_MIN_SIGNATURE_N=10
# ============================
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt,time
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score
from IPython.display import display
from google.colab import drive
if not Path("/content/drive/MyDrive").exists():drive.mount("/content/drive")
T12_BASE=Path("/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/04B_roi_quarterly_LOSO")
T12_SAMPLES=T12_BASE/"samples"
T12_FOLDS=T12_BASE/"folds"
T12_DIR=T12_BASE/"12_temporal_interpretation"
T12_CACHE=T12_DIR/"folds"
T12_CACHE.mkdir(parents=True,exist_ok=True)
# Source band order from NB04A: B2 B3 B4 B8 B5 B6 B7 B8A B11 B12, then clear count
T12_BANDS={"blue":1,"green":2,"red":3,"nir":4,"re1":5,"re2":6,"re3":7,"narrow_nir":8,"swir1":9,"swir2":10}
def t12_features(df):
    new=pd.DataFrame(index=df.index)
    quarters={}
    for q in range(4):
        base=q*11
        names={k:f"S2_QUARTERLY_3YR__{base+v:02d}" for k,v in T12_BANDS.items()}
        missing=[c for c in names.values() if c not in df.columns]
        if missing:raise KeyError(("Quarterly columns missing",missing))
        b={k:df[col].to_numpy(np.float64) for k,col in names.items()}
        nir=b["nir"];red=b["red"];green=b["green"];re=b["re1"];swir=b["swir1"];swir2=b["swir2"]
        # Reflectances are 0..1 (NB04A applies 1/10000 scale).
        eps=1e-6
        x={"NDVI":(nir-red)/(nir+red+eps),
           "NDMI":(nir-swir)/(nir+swir+eps),
           "NDRE":(b["narrow_nir"]-re)/(b["narrow_nir"]+re+eps),
           "NBR":(nir-swir2)/(nir+swir2+eps),
           "EVI2":2.5*(nir-red)/(nir+2.4*red+1.+eps)}
        for ix in T12_SEASONAL_INDICES:
            new[f"IDX_{ix}_Q{q+1}"]=x[ix]
        quarters[q+1]=x
    for ix in T12_SEASONAL_INDICES:
        v=np.stack([quarters[q][ix] for q in range(1,5)],axis=1)
        new[f"IDX_{ix}_annual_mean"]=v.mean(axis=1)
        new[f"IDX_{ix}_seasonal_range"]=np.ptp(v,axis=1)
        new[f"IDX_{ix}_seasonal_std"]=v.std(axis=1)
        new[f"IDX_{ix}_Q1_minus_Q3"]=v[:,0]-v[:,2]
        new[f"IDX_{ix}_Q2_minus_Q4"]=v[:,1]-v[:,3]
    arr=new.to_numpy(np.float64)
    if not np.isfinite(arr).all():raise ValueError("Non-finite derived index detected")
    # Reflectance sums close to zero would make indices unstable; audit explicitly.
    return new
def t12_metrics(d):
    y=d.observed.to_numpy(float);p=d.predicted.to_numpy(float)
    h=y>=.40
    return dict(n=len(d),R2=r2_score(y,p) if len(d)>1 and np.var(y)>1e-10 else np.nan,
                MAE_pp=100*np.mean(abs(y-p)),bias_pp=100*np.mean(p-y),
                n_high=int(h.sum()),high_bias_pp=100*np.mean((p-y)[h]) if h.any() else np.nan,
                high_recall=float(np.mean(p[h]>=.40)) if h.any() else np.nan)
allrows=[];correlations=[];signatures=[]
for scale in T12_SCALES:
    cache=T12_SAMPLES/f"roi_cells_{scale}m.parquet"
    if not cache.exists():raise FileNotFoundError(cache)
    df=pd.read_parquet(cache)
    derived=t12_features(df)
    emb=[c for c in df.columns if c.startswith("EMB_ANNUAL__")]
    assert emb,"Cached embedding predictors missing"
    derived_cols=list(derived.columns)
    frame=pd.concat([df,derived],axis=1)
    # Ecosystem interpretation: within-forest correlations avoid confounding by differences in means among sites.
    for site,part in frame.groupby("site"):
        for index in T12_SEASONAL_INDICES:
            for measure in ["annual_mean","seasonal_range","seasonal_std","Q1_minus_Q3","Q2_minus_Q4"]:
                col=f"IDX_{index}_{measure}"
                rho,pval=spearmanr(part[col],part.mean_probability)
                correlations.append(dict(scale_m=scale,forest=site,n=len(part),feature=col,
                    spearman_rho=rho,p_uncorrected=pval))
            for quarter in range(1,5):
                col=f"IDX_{index}_Q{quarter}"
                for group,subset in [("low_lt20",part[part.mean_probability<.20]),
                                     ("high_ge40",part[part.mean_probability>=.40])]:
                    if len(subset)<T12_MIN_SIGNATURE_N:continue
                    signatures.append(dict(scale_m=scale,forest=site,index=index,quarter=quarter,
                        abundance_group=group,n=len(subset),
                        mean=float(subset[col].mean()),median=float(subset[col].median()),
                        sd=float(subset[col].std())))
    for heldout in sorted(df.site.unique()):
        train=frame[frame.site!=heldout];test=frame[frame.site==heldout]
        for label in T12_NEW_MODELS:
            cols=derived_cols if label=="INDICES" else emb+derived_cols
            key=f"FOUR_FORESTS_{scale}m_{heldout}_{label}_mean_probability_w0"
            target=T12_CACHE/f"{key}.parquet"
            if target.exists() and not T12_FORCE:
                pred=pd.read_parquet(target)
                print("CACHE",key,flush=True)
            else:
                t=time.perf_counter()
                model=HistGradientBoostingRegressor(learning_rate=.06,max_iter=200,
                    max_leaf_nodes=20,min_samples_leaf=40,l2_regularization=2,
                    early_stopping=False,random_state=T12_SEED)
                model.fit(train[cols].to_numpy(np.float32),
                          train.mean_probability.to_numpy(np.float32))
                pred=test[["site","group","mosaic","row","col"]].copy()
                pred["observed"]=test.mean_probability.to_numpy(np.float32)
                pred["predicted"]=np.clip(model.predict(test[cols].to_numpy(np.float32)),0,1)
                pred.to_parquet(target,index=False)
                print("FIT",key,"sec",round(time.perf_counter()-t,1),flush=True)
            assert len(pred)==len(test),"Stale new-prediction cache; set T12_FORCE=True"
            for grp,part in pred.groupby("group"):
                allrows.append(dict(scale_m=scale,holdout=heldout,mosaic_group=grp,
                    model=label,**t12_metrics(part)))
        for label,w in T12_BASELINES:
            key=f"FOUR_FORESTS_{scale}m_{heldout}_{label}_mean_probability_w{w}"
            path=T12_FOLDS/f"{key}.parquet"
            if not path.exists():raise FileNotFoundError(f"Baseline fold not found: {path}")
            pred=pd.read_parquet(path)
            for grp,part in pred.groupby("group"):
                allrows.append(dict(scale_m=scale,holdout=heldout,mosaic_group=grp,
                    model=f"{label}_w{w}",**t12_metrics(part)))
    pd.DataFrame(allrows).to_csv(T12_DIR/"12_model_scores_progress.csv",index=False)
scores=pd.DataFrame(allrows)
scores.to_csv(T12_DIR/"12_model_scores_by_mosaic.csv",index=False)
corr=pd.DataFrame(correlations)
corr.to_csv(T12_DIR/"12_temporal_feature_associations.csv",index=False)
sig=pd.DataFrame(signatures)
sig.to_csv(T12_DIR/"12_low_high_seasonal_signatures.csv",index=False)
print("\nSTRICT FOREST HOLDOUT: per mosaic and predictors")
display(scores[scores.scale_m==75].round(3))
# Macro MAE across four forests (not across mosaics)
summ=[]
for (scale,label),part in scores.groupby(["scale_m","model"]):
    forest_mae=part.groupby("holdout").apply(
        lambda p:np.average(p.MAE_pp,weights=p.n),include_groups=False)
    mos=part[part.holdout=="MOS"]
    summ.append(dict(scale_m=scale,model=label,macro_MAE_pp=forest_mae.mean(),
        MOS_MAE_pp=np.average(mos.MAE_pp,weights=mos.n),
        MOS_high_recall=np.average(mos.high_recall,weights=mos.n_high),
        MOS_high_bias_pp=np.average(mos.high_bias_pp,weights=mos.n_high)))
summary=pd.DataFrame(summ).sort_values(["scale_m","macro_MAE_pp"])
summary.to_csv(T12_DIR/"12_summary.csv",index=False)
print("\n=== Temporal predictors vs cached baselines ===")
display(summary.round(3))
print("\n=== Within-forest index associations, top absolute Spearman rho (exploratory) ===")
top=(corr[corr.scale_m==75].assign(absrho=lambda x:x.spearman_rho.abs())
     .sort_values("absrho",ascending=False).groupby("forest").head(8))
display(top.round(3))
# Relative seasonal signature per forest, avoids misleading pooling across forests.
for site in sorted(sig.forest.unique()):
    d=sig[(sig.scale_m==75)&(sig.forest==site)&
          (sig["index"].isin(["NDVI","NDMI","NDRE"]))]
    if not {"low_lt20","high_ge40"}.issubset(set(d.abundance_group)):continue
    fig,ax=plt.subplots(1,3,figsize=(11,3.3),constrained_layout=True)
    for axis,index in zip(ax,["NDVI","NDMI","NDRE"]):
        for group,part in d[d["index"]==index].groupby("abundance_group"):
            axis.plot(part.quarter,part["mean"],"-o",label=group)
        axis.set(xlabel="Quarter (3-year pooled)",title=f"{site}: {index}",xticks=[1,2,3,4])
        axis.legend(fontsize=7)
    fig.savefig(T12_DIR/f"12_seasonal_signature_{site}.png",dpi=170)
    plt.show();plt.close(fig)
print("Saved Block 12:",T12_DIR,flush=True)


## 13 — Does seasonality add independent information beyond mean reflectance?

**Standalone** from the Block 07 cached ROI sample Parquets. Controls the comparison by holding **model algorithm, fitting hyperparameters, spatial test folds and training samples constant**. Tests five single-index means (EVI2/NDVI/NDRE/NDMI/NBR), five-index means, mean + seasonal contrasts, embeddings + index means, and embeddings + means + seasonal contrasts. Focus on Yoko and Yangambi for the Kisangani mapping objective, while preserving strict four-forest LOSO for MOS and Baego. EVI2-vs-Musanga figures include **approximate spatial-block bootstrap intervals for association** (not predictive uncertainty). All fitted fold predictions are saved and reused on rerun. **No new satellite extraction, UAV aggregation, or earlier notebook blocks required.** Note: 3-year pooled quarterly composites characterize average seasonality, not interannual trends.

In [ ]:
# BLOCK 13 — controlled seasonal incremental value, strict FOUR_FORESTS LOSO
# ===================== SETTINGS =====================
B13_SCALES=[75,100,125]
B13_INDICES=["EVI2","NDVI","NDRE","NDMI","NBR"]
B13_SEASONAL_TERMS=["seasonal_range","Q1_minus_Q3","Q2_minus_Q4"]
B13_BLOCK_M=400  # approximate bootstrap block width for descriptive EVI2 correlations
B13_BOOTSTRAPS=300
B13_SEED=20261022
B13_FORCE=False
# ====================================================
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt,time
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score
from IPython.display import display
from google.colab import drive
if not Path("/content/drive/MyDrive").exists(): drive.mount("/content/drive")
B13_ROOT=Path("/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/04B_roi_quarterly_LOSO")
B13_IN=B13_ROOT/"samples"
B13_OUT=B13_ROOT/"13_seasonality_incremental"
B13_FOLDS=B13_OUT/"folds"
B13_FOLDS.mkdir(parents=True,exist_ok=True)
B13_BAND_MAP={"blue":1,"green":2,"red":3,"nir":4,"re1":5,"re2":6,"re3":7,"nir_narrow":8,"swir1":9,"swir2":10}
def b13_engineer(source):
    derived=pd.DataFrame(index=source.index)
    all_quarters={ix:[] for ix in B13_INDICES}
    for q in range(4):
        v={k:source[f"S2_QUARTERLY_3YR__{q*11+b:02d}"].to_numpy(np.float64)
           for k,b in B13_BAND_MAP.items()}
        red=v["red"];nir=v["nir"];re=v["re1"];narrow=v["nir_narrow"];swir=v["swir1"];swir2=v["swir2"]
        e=1e-6
        ixvals={
            "EVI2":2.5*(nir-red)/(nir+2.4*red+1+e),
            "NDVI":(nir-red)/(nir+red+e),
            "NDRE":(narrow-re)/(narrow+re+e),
            "NDMI":(nir-swir)/(nir+swir+e),
            "NBR":(nir-swir2)/(nir+swir2+e),
        }
        for ix in B13_INDICES: all_quarters[ix].append(ixvals[ix])
    for ix in B13_INDICES:
        x=np.stack(all_quarters[ix],axis=1)
        derived[f"{ix}_mean"]=np.mean(x,axis=1)
        derived[f"{ix}_seasonal_range"]=np.ptp(x,axis=1)
        derived[f"{ix}_Q1_minus_Q3"]=x[:,0]-x[:,2]
        derived[f"{ix}_Q2_minus_Q4"]=x[:,1]-x[:,3]
    assert np.isfinite(derived.to_numpy(float)).all(),"Non-finite seasonal features"
    return derived
def b13_scores(part):
    y=part.observed.to_numpy(float);p=part.predicted.to_numpy(float)
    high=y>=.40;low=y<.20
    return dict(n=len(y),R2=r2_score(y,p) if np.var(y)>1e-12 and len(y)>1 else np.nan,
                MAE_pp=100*np.mean(abs(y-p)),bias_pp=100*np.mean(p-y),
                n_high=int(high.sum()),
                high_bias_pp=100*np.mean((p-y)[high]) if high.any() else np.nan,
                high_recall=float(np.mean(p[high]>=.40)) if high.any() else np.nan,
                low_MAE_pp=100*np.mean(abs((p-y)[low])) if low.any() else np.nan)
def b13_block_rho(d,idx,scale,rng):
    # Block bootstrap resamples entire spatial blocks, not cells independently.
    d=d[["row","col","mean_probability",idx]].dropna()
    coords=np.column_stack((np.floor(d.row.to_numpy()*scale/B13_BLOCK_M).astype(int),
                            np.floor(d.col.to_numpy()*scale/B13_BLOCK_M).astype(int)))
    _,inv=np.unique(coords,axis=0,return_inverse=True)
    blocks=[np.where(inv==i)[0] for i in range(inv.max()+1)]
    rho=float(spearmanr(d[idx],d.mean_probability).statistic)
    if len(blocks)<6:return rho,np.nan,np.nan,len(blocks)
    values=[]
    x=d[idx].to_numpy(float);y=d.mean_probability.to_numpy(float)
    for _ in range(B13_BOOTSTRAPS):
        sample=np.concatenate([blocks[i] for i in rng.integers(len(blocks),size=len(blocks))])
        if len(np.unique(x[sample]))>1 and len(np.unique(y[sample]))>1:
            values.append(spearmanr(x[sample],y[sample]).statistic)
    ci=np.quantile(values,[.025,.975]) if len(values)>=20 else [np.nan,np.nan]
    return rho,float(ci[0]),float(ci[1]),len(blocks)
B13_GROUPS=["Baego","MOS","Yangambi","Yoko"]
B13_ROWS=[];B13_CORR=[];B13_PLOT_TABLE=[]
for scale in B13_SCALES:
    data_path=B13_IN/f"roi_cells_{scale}m.parquet"
    if not data_path.is_file():raise FileNotFoundError(f"Missing Block 07 cache: {data_path}")
    data=pd.read_parquet(data_path).reset_index(drop=True)
    assert set(data.site.unique())==set(B13_GROUPS),"Expected four forests"
    engineered=b13_engineer(data)
    data=pd.concat([data,engineered],axis=1)
    emb=[c for c in data.columns if c.startswith("EMB_ANNUAL__")]
    assert len(emb)>0,"Missing annual embedding columns"
    mean_cols=[f"{ix}_mean" for ix in B13_INDICES]
    season_cols=[f"{ix}_{suffix}" for ix in B13_INDICES for suffix in B13_SEASONAL_TERMS]
    feature_sets={f"MEAN_{ix}":[f"{ix}_mean"] for ix in B13_INDICES}
    feature_sets.update({
        "MEANS_5":mean_cols,
        "MEANS_5_PLUS_SEASON":mean_cols+season_cols,
        "EMB_PLUS_MEANS":emb+mean_cols,
        "EMB_PLUS_MEANS_SEASON":emb+mean_cols+season_cols
    })
    for site,part in data.groupby("site"):
        for ix in B13_INDICES:
            rng=np.random.default_rng(B13_SEED+scale+sum(map(ord,site))+sum(map(ord,ix)))
            rho,lo,hi,nblocks=b13_block_rho(part,f"{ix}_mean",scale,rng)
            B13_CORR.append(dict(scale_m=scale,site=site,index=ix,n_cells=len(part),
                                 n_spatial_blocks=nblocks,spearman_rho=rho,
                                 spatial_bootstrap_CI_low=lo,spatial_bootstrap_CI_high=hi))
    if scale==75:
        B13_PLOT_TABLE.append(data[["site","group","mean_probability","EVI2_mean","EVI2_seasonal_range"]].copy())
    for holdout in B13_GROUPS:
        train=data[data.site!=holdout];test=data[data.site==holdout]
        for name,features in feature_sets.items():
            key=f"FOUR_FORESTS_{scale}m_{holdout}_{name}"
            saved=B13_FOLDS/f"{key}.parquet"
            if saved.exists() and not B13_FORCE:
                pred=pd.read_parquet(saved)
                assert len(pred)==len(test) and np.allclose(
                    pred.observed.to_numpy(),test.mean_probability.to_numpy(),atol=1e-6),(
                    "Stale cached predictions: set B13_FORCE=True",saved)
                print("CACHE",key,flush=True)
            else:
                t0=time.perf_counter()
                reg=HistGradientBoostingRegressor(learning_rate=.06,max_iter=200,
                    max_leaf_nodes=20,min_samples_leaf=40,l2_regularization=2,
                    early_stopping=False,random_state=B13_SEED)
                reg.fit(train[features].to_numpy(np.float32),
                        train.mean_probability.to_numpy(np.float32))
                pred=test[["site","group","mosaic","row","col"]].copy()
                pred["observed"]=test.mean_probability.to_numpy(np.float32)
                pred["predicted"]=np.clip(reg.predict(test[features].to_numpy(np.float32)),0,1)
                pred.to_parquet(saved,index=False)
                print("FIT",key,"sec",round(time.perf_counter()-t0,1),flush=True)
            for mosaic,part in pred.groupby("group"):
                B13_ROWS.append(dict(scale_m=scale,forest_holdout=holdout,
                    mosaic_group=mosaic,model=name,**b13_scores(part)))
    pd.DataFrame(B13_ROWS).to_csv(B13_OUT/"13_per_mosaic_progress.csv",index=False)
metrics=pd.DataFrame(B13_ROWS)
metrics.to_csv(B13_OUT/"13_per_mosaic_metrics.csv",index=False)
corr=pd.DataFrame(B13_CORR)
corr.to_csv(B13_OUT/"13_spatial_block_EVI_index_correlations.csv",index=False)
summary=[]
for (scale,name),group in metrics.groupby(["scale_m","model"]):
    # Equal-FOREST MAE is weighted within forests, then averaged across four forests.
    forest_mae=group.groupby("forest_holdout").apply(
        lambda x:np.average(x.MAE_pp,weights=x.n),include_groups=False)
    mos=group[group.forest_holdout=="MOS"]
    y=[];p=[];forest=[]
    for held in B13_GROUPS:
        q=pd.read_parquet(B13_FOLDS/f"FOUR_FORESTS_{scale}m_{held}_{name}.parquet")
        y.append(q.observed.to_numpy(float));p.append(q.predicted.to_numpy(float))
        forest.extend([held]*len(q))
    y=np.concatenate(y);p=np.concatenate(p)
    forest=np.array(forest)
    sizes=pd.Series(forest).value_counts()
    w=np.array([1/sizes[v] for v in forest]);w=w/w.sum()
    ybar=np.dot(w,y);sst=np.dot(w,(y-ybar)**2);mse=np.dot(w,(y-p)**2)
    site_means={s:np.mean(y[forest==s]) for s in B13_GROUPS}
    within=np.dot(w,(y-np.array([site_means[s] for s in forest]))**2)
    summary.append(dict(scale_m=scale,model=name,pooled_R2=r2_score(y,p),
        balanced_R2=1-mse/sst,within_forest_R2=1-mse/within,
        macro_MAE_pp=forest_mae.mean(),
        Yoko_MAE_pp=float(group[group.forest_holdout=="Yoko"].MAE_pp.iloc[0]),
        Yangambi_MAE_pp=float(group[group.forest_holdout=="Yangambi"].MAE_pp.iloc[0]),
        MOS_MAE_pp=np.average(mos.MAE_pp,weights=mos.n),
        MOS_high_recall=np.average(mos.high_recall,weights=mos.n_high),
        MOS_high_bias_pp=np.average(mos.high_bias_pp,weights=mos.n_high)))
overview=pd.DataFrame(summary)
overview.to_csv(B13_OUT/"13_model_summary.csv",index=False)
print("\n=== Controlled seasonal incremental-value experiment (strict forest LOSO) ===")
display(overview.sort_values(["scale_m","macro_MAE_pp"]).round(3))
print("\n=== Direct increment: adding seasonality to index MEANS (identical models/folds) ===")
pairs=[("MEANS_5","MEANS_5_PLUS_SEASON"),("EMB_PLUS_MEANS","EMB_PLUS_MEANS_SEASON")]
comparison=[]
for scale in B13_SCALES:
    d=overview[overview.scale_m==scale].set_index("model")
    for baseline,seasonal in pairs:
        a=d.loc[baseline];b=d.loc[seasonal]
        comparison.append(dict(scale_m=scale,baseline=baseline,seasonal=seasonal,
            delta_macro_MAE_pp=b.macro_MAE_pp-a.macro_MAE_pp,
            delta_balanced_R2=b.balanced_R2-a.balanced_R2,
            delta_Yoko_MAE_pp=b.Yoko_MAE_pp-a.Yoko_MAE_pp,
            delta_Yangambi_MAE_pp=b.Yangambi_MAE_pp-a.Yangambi_MAE_pp,
            delta_MOS_high_recall=b.MOS_high_recall-a.MOS_high_recall))
comp=pd.DataFrame(comparison)
comp.to_csv(B13_OUT/"13_added_value_of_seasonality.csv",index=False)
display(comp.round(3))
print("\n=== EVI2/other index correlations: approximate block-bootstrap CIs, not causal ===")
display(corr[(corr.scale_m==75)&(corr["index"]=="EVI2")].round(3))
# EVI2 association: descriptive binned scatter by forest, not raw-cell scatter clutter.
plot_df=B13_PLOT_TABLE[0]
fig,ax=plt.subplots(2,2,figsize=(10,8),constrained_layout=True)
for axis,site in zip(ax.flat,B13_GROUPS):
    d=plot_df[plot_df.site==site].copy()
    axis.scatter(d.EVI2_mean,100*d.mean_probability,s=3,alpha=.12,rasterized=True)
    quant=pd.qcut(d.EVI2_mean,q=min(12,max(4,len(d)//20)),duplicates="drop")
    means=d.groupby(quant,observed=True).agg(evi=("EVI2_mean","mean"),musanga=("mean_probability","mean"),n=("mean_probability","size"))
    axis.plot(means.evi,100*means.musanga,"o-",linewidth=2,label="Binned mean")
    row=corr[(corr.scale_m==75)&(corr.site==site)&(corr["index"]=="EVI2")].iloc[0]
    axis.set(title=f"{site} | Spearman rho={row.spearman_rho:.2f} [{row.spatial_bootstrap_CI_low:.2f}, {row.spatial_bootstrap_CI_high:.2f}]",
             xlabel="3-year seasonal-mean EVI2",ylabel="UAV-derived Musanga probability (%)")
    axis.legend(fontsize=7)
fig.savefig(B13_OUT/"13_EVI2_Musanga_by_forest_75m.png",dpi=180)
plt.show();plt.close(fig)
print("Outputs:",B13_OUT,flush=True)


## 14 — Kisangani validation, uncertainty, and circle representativeness

Run Blocks 14A and 14B independently using cached data. Block 14A compares complete ROI UAV cover distributions against exhaustive annotation circle footprints. Block 14B uses disjoint spatial train/calibration/test blocks in Yoko and Yangambi; non-Kisangani sites supplement training only. Estimated 90% intervals have empirical coverage diagnostics, not guarantees of spatial-transfer coverage.

In [ ]:
# BLOCK 14A — Full ROI vs annotation-circle UAV cover distribution (10 m)
from pathlib import Path
import numpy as np,pandas as pd,geopandas as gpd,rasterio,matplotlib.pyplot as plt,json
from rasterio.features import geometry_mask
from IPython.display import display
from google.colab import drive
if not Path("/content/drive/MyDrive").exists():drive.mount("/content/drive")
R=Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
AGG=R/"Final/Outputs/V3_1/04_satellite_full_warmstart/cache/aggregates"
OUT=R/"Final/Outputs/V3_1/04B_roi_quarterly_LOSO/14_kisangani_uncertainty"
OUT.mkdir(parents=True,exist_ok=True)
ROI=R/"data/Orthoo_RGB/ROI.shp"
CIRC=R/"data/Orthoo_RGB/Circles_ROI_NAdine.shp"
MAN=R/"Final/Outputs/V3/00_input_audit/00_input_manifest.json"
if MAN.exists():
    manifest=json.loads(MAN.read_text())
    if not ROI.exists(): ROI=Path(manifest["roi"])
    if not CIRC.exists():
        for k,v in manifest.items():
            if "circle" in k.lower() and "roi" in k.lower() and isinstance(v,str) and Path(v).exists():
                CIRC=Path(v);break
assert ROI.is_file() and CIRC.is_file(),(ROI,CIRC)
roi=gpd.read_file(ROI);circles=gpd.read_file(CIRC)
assert roi.crs and circles.crs
names={"Baego":"Baego.tif","MOS1_2":"Mos_1_2_transparent_mosaic_group1_georef.tif",
       "MOS5_6":"Mos_5_6_transparent_mosaic_group1_georef.tif",
       "Yangambi":"Yangambi_georef.tif","Yoko":"Yoko_kvo_georef.tif"}
edges=np.array([0,.05,.1,.2,.3,.4,.5,.6,.75,1.000001])
rows=[];counts=[]
fig,axes=plt.subplots(2,3,figsize=(12,8),constrained_layout=True)
for ax,(site,name) in zip(axes.flat,names.items()):
    path=AGG/f"{Path(name).stem}_10m.tif"
    assert path.is_file(),path
    with rasterio.open(path) as ds:
        p,_,valid=ds.read()
        roi_mask=geometry_mask(roi.to_crs(ds.crs).geometry,out_shape=p.shape,
                               transform=ds.transform,invert=True)
        circ_mask=geometry_mask(circles.to_crs(ds.crs).geometry,out_shape=p.shape,
                                transform=ds.transform,invert=True)
        eligible=roi_mask&np.isfinite(p)&(p>=0)&(p<=1)&(valid>=.9)
        for label,sel in {"ROI_all":eligible,"Inside_circles":eligible&circ_mask,
                          "Outside_circles":eligible&~circ_mask}.items():
            x=p[sel];x=x[np.isfinite(x)]
            rows.append(dict(mosaic=site,population=label,n=len(x),
                mean_pp=100*np.mean(x) if len(x) else np.nan,
                median_pp=100*np.median(x) if len(x) else np.nan,
                p90_pp=100*np.quantile(x,.9) if len(x) else np.nan,
                pct_ge20=100*np.mean(x>=.2) if len(x) else np.nan,
                pct_ge40=100*np.mean(x>=.4) if len(x) else np.nan))
            bins=np.histogram(x,bins=edges)[0]
            counts.extend(dict(mosaic=site,population=label,lower=edges[i],upper=edges[i+1],n=int(n))
                          for i,n in enumerate(bins))
            if label!="Outside_circles" and len(x):
                ax.hist(x*100,bins=edges*100,histtype="step",linewidth=2,
                        weights=np.ones(len(x))/len(x),label=f"{label} n={len(x)}")
        ax.set(title=site,xlabel="UAV-derived Musanga probability (%)",ylabel="Share of valid 10 m cells")
        ax.legend(fontsize=7)
axes.flat[-1].axis("off")
res=pd.DataFrame(rows)
res.to_csv(OUT/"14A_UAV_cover_vs_circle_footprints.csv",index=False)
pd.DataFrame(counts).to_csv(OUT/"14A_UAV_cover_histograms.csv",index=False)
fig.savefig(OUT/"14A_cover_comparison.png",dpi=160)
plt.show();plt.close(fig)
display(res.round(2))
print("Circle distribution uses UAV-model probabilities inside circle footprints; it is NOT manually digitized crown cover.",OUT)


In [ ]:
# BLOCK 14B — spatially disjoint train, calibration, test; Yoko + Yangambi
# Edit settings below. This cell is standalone; read cached Block 07 samples.
SCALES=[75,100]; REPEATS=3; BLOCK_M=400; BUFFER_M=150
TEST_FRACTION=.20; CAL_FRACTION=.20; LEVEL=.90; FORCE=False
MODELS={"EMB":["EMB_ANNUAL"],"EMB_QUARTERLY":["EMB_ANNUAL","S2_QUARTERLY_3YR"]}
import numpy as np,pandas as pd,matplotlib.pyplot as plt,time
from pathlib import Path
from scipy.spatial import cKDTree
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import r2_score
from IPython.display import display
from google.colab import drive
if not Path("/content/drive/MyDrive").exists():drive.mount("/content/drive")
BASE=Path("/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/04B_roi_quarterly_LOSO")
OUTPUT=BASE/"14_kisangani_uncertainty";FOLD=OUTPUT/"folds";FOLD.mkdir(parents=True,exist_ok=True)
def split_regions(df,scale,rep):
    rng=np.random.default_rng(20261022+scale*10+rep)
    training=[df[~df.site.isin(["Yoko","Yangambi"])]]
    calibrations=[];testing=[]
    for site in ["Yoko","Yangambi"]:
        d=df[df.site==site].copy()
        d["px"]=d.col.to_numpy(float)*scale
        d["py"]=d.row.to_numpy(float)*scale
        d["sp_block"]=((d.px//BLOCK_M).astype(int).astype(str)+"_"+
                       (d.py//BLOCK_M).astype(int).astype(str))
        groups=d.groupby("sp_block").indices
        ids=list(groups);rng.shuffle(ids)
        picked={"cal":[],"test":[]};i=0
        for key,frac in [("test",TEST_FRACTION),("cal",CAL_FRACTION)]:
            count=0
            while i<len(ids) and count<frac*len(d):
                g=ids[i];i+=1;picked[key].append(g);count+=len(groups[g])
        t=d[d.sp_block.isin(picked["test"])].copy()
        c=d[d.sp_block.isin(picked["cal"])].copy()
        tr=d[~d.sp_block.isin(picked["test"]+picked["cal"])].copy()
        def separated(a,b):
            if a.empty or b.empty:return a.iloc[0:0].copy()
            dist=cKDTree(b[["px","py"]].to_numpy()).query(a[["px","py"]].to_numpy())[0]
            return a.iloc[np.flatnonzero(dist>=BUFFER_M)].copy()
        c=separated(c,tr)
        t=separated(t,tr)
        t=separated(t,c)
        assert len(c)>=8 and len(t)>=8,(site,scale,rep,len(c),len(t))
        training.append(tr);calibrations.append(c);testing.append(t)
    return (pd.concat(training,ignore_index=True),pd.concat(calibrations,ignore_index=True),
            pd.concat(testing,ignore_index=True))
def metric(part):
    y=part.observed.to_numpy(float);p=part.predicted.to_numpy(float)
    high=y>=.4
    return dict(n=len(part),R2=r2_score(y,p) if len(y)>1 and np.var(y)>1e-10 else np.nan,
                MAE_pp=100*np.mean(abs(y-p)),bias_pp=100*np.mean(p-y),
                interval_coverage=np.mean((y>=part.lower90)&(y<=part.upper90)),
                interval_width_pp=100*np.mean(part.upper90-part.lower90),
                high_n=int(high.sum()),
                high_interval_coverage=np.mean((y[high]>=part.lower90.to_numpy()[high])&
                       (y[high]<=part.upper90.to_numpy()[high])) if high.any() else np.nan,
                novel_fraction=np.mean(part.novel))
all_scores=[];support=[]
for scale in SCALES:
    df=pd.read_parquet(BASE/"samples"/f"roi_cells_{scale}m.parquet").reset_index(drop=True)
    for repeat in range(REPEATS):
        tr,ca,te=split_regions(df,scale,repeat)
        support.append(dict(scale_m=scale,repeat=repeat,n_training=len(tr),n_cal=len(ca),
                            n_test=len(te),Yoko_test=int((te.site=="Yoko").sum()),
                            Yangambi_test=int((te.site=="Yangambi").sum())))
        print("SPLIT",support[-1],flush=True)
        for name,families in MODELS.items():
            cols=[c for c in df.columns if any(c.startswith(f+"__") for f in families)]
            assert cols,"Missing predictors "+name
            tag=f"{scale}m_repeat{repeat}_{name}"
            path=FOLD/f"{tag}.parquet"
            if path.exists() and not FORCE:
                result=pd.read_parquet(path)
                assert len(result)==len(te) and np.allclose(result.observed,te.mean_probability,atol=1e-6),(
                    "Stale cached fold; set FORCE=True",path)
                print("CACHE",tag,flush=True)
            else:
                start=time.perf_counter()
                X=tr[cols].to_numpy(np.float32)
                model=HistGradientBoostingRegressor(learning_rate=.06,max_iter=200,
                    max_leaf_nodes=20,min_samples_leaf=40,l2_regularization=2,
                    early_stopping=False,random_state=20261022)
                model.fit(X,tr.mean_probability.to_numpy(np.float32))
                pc=np.clip(model.predict(ca[cols].to_numpy(np.float32)),0,1)
                pt=np.clip(model.predict(te[cols].to_numpy(np.float32)),0,1)
                residual=np.abs(ca.mean_probability.to_numpy(float)-pc)
                n=len(residual);rank=min(n,int(np.ceil((n+1)*LEVEL)))
                radius=np.partition(residual,rank-1)[rank-1]
                # Exploratory predictor-space novelty, not a validated applicability threshold.
                scaler=StandardScaler().fit(X)
                Z=scaler.transform(X);ZT=scaler.transform(te[cols].to_numpy(np.float32))
                pca=PCA(n_components=min(10,X.shape[1],len(X)-1),random_state=20261022).fit(Z)
                Z=pca.transform(Z);ZT=pca.transform(ZT)
                nbrs=NearestNeighbors(n_neighbors=2).fit(Z)
                self_d=nbrs.kneighbors(Z)[0][:,1]
                test_d=nbrs.kneighbors(ZT,n_neighbors=1)[0][:,0]
                threshold=np.quantile(self_d,.95)
                result=te[["site","group","mosaic","row","col"]].copy()
                result["observed"]=te.mean_probability.to_numpy(float)
                result["predicted"]=pt
                result["lower90"]=np.maximum(0,pt-radius)
                result["upper90"]=np.minimum(1,pt+radius)
                result["novel"]=test_d>threshold
                result["novelty_distance"]=test_d
                result.to_parquet(path,index=False)
                print("FIT",tag,"interval radius pp",round(100*radius,2),
                      "seconds",round(time.perf_counter()-start,1),flush=True)
            for site,part in result.groupby("site"):
                all_scores.append(dict(scale_m=scale,repeat=repeat,model=name,site=site,**metric(part)))
    pd.DataFrame(all_scores).to_csv(OUTPUT/"14B_progress.csv",index=False)
results=pd.DataFrame(all_scores)
results.to_csv(OUTPUT/"14B_spatial_test_metrics.csv",index=False)
pd.DataFrame(support).to_csv(OUTPUT/"14B_split_support.csv",index=False)
summary=results.groupby(["scale_m","site","model"],as_index=False).agg(
    R2_mean=("R2","mean"),R2_sd=("R2","std"),MAE_pp=("MAE_pp","mean"),
    bias_pp=("bias_pp","mean"),interval_coverage=("interval_coverage","mean"),
    interval_width_pp=("interval_width_pp","mean"),
    n_high_mean=("high_n","mean"),high_interval_coverage=("high_interval_coverage","mean"),
    novel_fraction=("novel_fraction","mean"))
summary.to_csv(OUTPUT/"14B_summary.csv",index=False)
display(summary.round(3))
print("Prediction interval coverage is empirical, spatially blocked and domain-limited.",OUTPUT)
